# TRIED Corpus Extractor

This notebook builds `corpus_train.jsonl`, the seed corpus of standalone PyTorch functions that the agent loop can later translate to Triton. It is intentionally conservative: rows that need hidden state, custom metadata, stochastic behavior, or unsupported dtypes are skipped instead of being forced into the tensor-only schema.


## Notebook Contract

Run this notebook on a CUDA Colab runtime. The output is still a corpus of PyTorch source examples, not the final attempt dataset used for SFT/DPO. Before locking a production corpus, enable the Inductor sanity check and review the manifest warnings.


In [ ]:
# Colab dependency setup. Keep this cell small so failures are easy to diagnose.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q transformers timm

import ast
import hashlib
import json
import math
import traceback
import uuid
from collections import Counter, defaultdict
from pathlib import Path
from textwrap import dedent

import torch

SEED = 42
DEVICE = 'cuda'
TARGET_TRAIN_RECORDS = 200
RUN_INDUCTOR_SANITY = True  # Enable before locking a final corpus; slower but required by the experiment protocol.
STRICT_EXPORT = True        # Set True to block export when quality warnings exist.

assert torch.cuda.is_available(), 'Switch to GPU: Runtime > Change runtime type > T4 GPU'
torch.manual_seed(SEED)
torch.set_grad_enabled(False)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'Colab runtime: {IN_COLAB}')


## 1. Fetch TritonBench

The extraction code reads only `tritonbench/tritonbench/operators`. Triton kernels in TritonBench are not indexed or copied into prompts; this notebook extracts or writes clean PyTorch reference functions only.


In [ ]:
TRITONBENCH_ROOT = Path('tritonbench')
OPERATORS_DIR = TRITONBENCH_ROOT / 'tritonbench' / 'operators'

if not OPERATORS_DIR.exists():
    !git clone -q --depth=1 https://github.com/meta-pytorch/tritonbench.git tritonbench
else:
    print(f'Using existing checkout: {TRITONBENCH_ROOT.resolve()}')

assert OPERATORS_DIR.exists(), f'Missing TritonBench operators directory: {OPERATORS_DIR}'
print(f'TritonBench operators: {len([p for p in OPERATORS_DIR.iterdir() if p.is_dir()])}')


## 2. Extraction Policy

These tables define which TritonBench operators can become tensor-only corpus rows, which categories they map to, and which shapes/dtypes are used for validation. Unsupported operators are skipped with explicit reasons so the gap is visible in `skipped.jsonl`.


In [ ]:
# Operators that cannot produce valid training examples
SKIP_OPS = {
    'launch_latency',    # measures overhead, not an op
    'template_attention',# placeholder
    'test_op',           # test
    'blackwell_attentions', # requires Blackwell GPU hardware
    'fp8_gemm', 'fp8_gemm_blockwise', 'fp8_gemm_rowwise',
    'fp8_gemm_rowwise_grouped', 'fp8_fused_quant_gemm_rowwise',
    'fp8_attention',     # fp8 dtype not in schema enum
    'bf16xint16_gemm',   # non-standard dtype combination
    'fp32_to_mx4', 'mx4_to_fp32', 'nvfp4_gemm',  # proprietary quantization formats
    'mixed_gemm',        # requires special mixed dtype packing beyond simple synthetic inputs
}

CATEGORY_MAP = {
    'addmm': 'matmul', 'gemm': 'matmul', 'grouped_gemm': 'matmul',
    'int4_gemm': 'matmul', 'mixed_gemm': 'matmul', 'gather_gemv': 'matmul',
    'flash_attention': 'fused_attention', 'flex_attention': 'fused_attention',
    'ragged_attention': 'fused_attention', 'decoding_attention': 'fused_attention',
    'gdpa': 'fused_attention', 'custom_shape_attentions': 'fused_attention',
    'layer_norm': 'normalization', 'rms_norm': 'normalization',
    'jagged_layer_norm': 'normalization', 'gdn_fwd_h': 'normalization',
    'inductor_residual_rmsnorm': 'normalization',
    'softmax': 'reduction', 'sum': 'reduction', 'welford': 'reduction',
    'jagged_softmax': 'reduction', 'jagged_mean': 'reduction',
    'jagged_sum': 'reduction', 'inductor_softmax_fused': 'reduction',
    'geglu': 'activation', 'swiglu': 'activation',
    'inductor_fused_linear_gelu': 'activation',
    'cross_entropy': 'loss', 'kl_div': 'loss', 'jsd': 'loss',
    'fused_linear_cross_entropy': 'loss', 'fused_linear_jsd': 'loss',
    'embedding': 'embedding',
    'rope': 'elementwise_chain', 'vector_add': 'elementwise_chain',
    'vector_exp': 'elementwise_chain', 'low_mem_dropout': 'elementwise_chain',
    'mamba2_chunk_scan': 'other', 'mamba2_chunk_state': 'other',
}

# Per-operator representative shapes — best effort, refined during validation
OPERATOR_SHAPES = {
    'softmax':              ([[4096, 1024]],                           ['float16']),
    'gemm':                 ([[512, 512], [512, 512]],                 ['float16', 'float16']),
    'addmm':                ([[512], [512, 512], [512, 512]],          ['float32', 'float32', 'float32']),
    'int4_gemm':            ([[512, 512], [512, 512]],                 ['int8', 'int8']),
    'mixed_gemm':           ([[512, 512], [512, 512]],                 ['float16', 'int8']),
    'grouped_gemm':         ([[4, 128, 512], [4, 512, 256]],          ['float16', 'float16']),
    'gather_gemv':          ([[512, 512], [512]],                      ['float16', 'float16']),
    'flash_attention':      ([[2, 12, 128, 64]] * 3,                  ['float16'] * 3),
    'flex_attention':       ([[2, 12, 128, 64]] * 3,                  ['float16'] * 3),
    'ragged_attention':     ([[2, 12, 128, 64]] * 3,                  ['float16'] * 3),
    'decoding_attention':   ([[2, 12, 1, 64], [2, 12, 128, 64], [2, 12, 128, 64]], ['float16'] * 3),
    'gdpa':                 ([[2, 12, 128, 64]] * 3,                  ['float16'] * 3),
    'custom_shape_attentions': ([[2, 12, 128, 64]] * 3,               ['float16'] * 3),
    'layer_norm':           ([[32, 128, 768], [768], [768]],          ['float32', 'float32', 'float32']),
    'rms_norm':             ([[32, 128, 4096], [4096]],               ['float32', 'float32']),
    'jagged_layer_norm':    ([[1024, 768], [768], [768]],             ['float32', 'float32', 'float32']),
    'gdn_fwd_h':            ([[32, 64, 64, 128]],                     ['float32']),
    'inductor_residual_rmsnorm': ([[32, 128, 768], [768]],            ['float32', 'float32']),
    'softmax':              ([[4096, 1024]],                           ['float16']),
    'sum':                  ([[4096, 1024]],                           ['float32']),
    'welford':              ([[4096, 1024]],                           ['float32']),
    'jagged_softmax':       ([[1024, 512]],                            ['float16']),
    'jagged_mean':          ([[1024, 512]],                            ['float32']),
    'jagged_sum':           ([[1024, 512]],                            ['float32']),
    'inductor_softmax_fused': ([[4096, 1024]],                        ['float16']),
    'geglu':                ([[32, 128, 4096], [32, 128, 4096]],      ['float16', 'float16']),
    'swiglu':               ([[32, 128, 4096], [32, 128, 4096]],      ['float16', 'float16']),
    'inductor_fused_linear_gelu': ([[32, 128, 768]],                  ['float32']),
    'cross_entropy':        ([[1024, 50257], [1024]],                 ['float32', 'int64']),
    'kl_div':               ([[1024, 50257], [1024, 50257]],          ['float32', 'float32']),
    'jsd':                  ([[1024, 50257], [1024, 50257]],          ['float32', 'float32']),
    'fused_linear_cross_entropy': ([[1024, 768], [50257, 768], [1024]], ['float32', 'float32', 'int64']),
    'fused_linear_jsd':     ([[1024, 768], [50257, 768], [1024, 50257]], ['float32', 'float32', 'float32']),
    'embedding':            ([[32, 128], [50257, 768]],               ['int64', 'float16']),
    'rope':                 ([[32, 32, 128, 128], [128, 64], [128, 64]], ['float32', 'float32', 'float32']),
    'vector_add':           ([[1048576], [1048576]],                   ['float32', 'float32']),
    'vector_exp':           ([[1048576]],                              ['float32']),
    'low_mem_dropout':      ([[32, 128, 768]],                        ['float32']),
    'mamba2_chunk_scan':    ([[32, 8, 64, 16]],                       ['float32']),
    'mamba2_chunk_state':   ([[32, 8, 64, 16]],                       ['float32']),
}

# These TritonBench baselines are stateful or benchmark-shaped, so the next
# cell provides clean standalone PyTorch equivalents instead of AST extraction.
CURATED_TRITONBENCH_OPS = {
    'cross_entropy', 'embedding', 'flash_attention', 'fused_linear_cross_entropy',
    'fused_linear_jsd', 'gather_gemv', 'geglu', 'gemm', 'grouped_gemm',
    'inductor_fused_linear_gelu', 'inductor_residual_rmsnorm',
    'inductor_softmax_fused', 'jsd', 'kl_div', 'layer_norm',
    'rms_norm', 'rope', 'sum', 'swiglu', 'vector_exp', 'welford',
}

# These need structured metadata/state that our schema cannot express as plain
# tensor-only inputs yet. Keep them out of manual review unless we decide to
# add dedicated synthetic input generators for them.
UNSUPPORTED_TRITONBENCH_OPS = {
    'decoding_attention': 'requires cache_seqlens and KV-cache layout metadata',
    'flex_attention': 'requires block mask / score modifier metadata',
    'gdn_fwd_h': 'requires recurrent state tensors and chunk sizing',
    'gdpa': 'requires jagged and padded data layout metadata',
    'jagged_layer_norm': 'requires seqlen and sparsity metadata',
    'jagged_mean': 'requires B/M/seqlen/sparsity metadata',
    'jagged_softmax': 'requires B/M/seqlen/sparsity metadata',
    'jagged_sum': 'requires B/M/seqlen/sparsity metadata',
    'low_mem_dropout': 'stochastic dropout cannot be represented as a deterministic tensor-only corpus row',
    'mamba2_chunk_scan': 'requires multiple recurrent state tensors',
    'mamba2_chunk_state': 'requires recurrent state tensors',
    'ragged_attention': 'requires ragged sequence metadata',
}

ALLOWED_SPLITS = {'train', 'eval'}
ALLOWED_DTYPES = {'float16', 'float32', 'bfloat16', 'float64', 'int8', 'int16', 'int32', 'int64', 'bool'}
ALLOWED_CATEGORIES = {
    'elementwise_chain', 'reduction', 'matmul', 'fused_attention',
    'normalization', 'activation', 'loss', 'embedding', 'convolution', 'other',
}
REQUIRED_RECORD_KEYS = {
    'example_id', 'split', 'origin', 'op_category', 'difficulty',
    'pytorch_code', 'input_shapes', 'input_dtypes', 'rng_seed',
}


# Selection targets are intentionally category-aware. The notebook can produce
# fewer than these numbers, but the quota report makes coverage gaps explicit.
CATEGORY_QUOTAS = {
    'matmul': 40,
    'normalization': 25,
    'reduction': 25,
    'elementwise_chain': 25,
    'fused_attention': 20,
    'activation': 20,
    'loss': 15,
    'embedding': 15,
    'convolution': 20,
    'other': 5,
}
CATEGORY_MIN_TARGETS = {
    'reduction': 8,
    'elementwise_chain': 8,
    'fused_attention': 5,
    'embedding': 5,
}
MAX_ROWS_PER_BEHAVIOR_FINGERPRINT = 3
SOURCE_PRIORITY = {
    'curated': 0,
    'tritonbench': 1,
    'hf': 2,
    'timm': 3,
}


## 3. Shared Record Helpers

Records get deterministic UUIDv5 identifiers based on origin, code, shapes, and dtypes. This makes reruns stable and keeps later agent-loop records joinable across experiments.


In [ ]:
# AST helpers

def find_benchmark_class(tree):
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            for base in node.bases:
                name = base.id if isinstance(base, ast.Name) else getattr(base, 'attr', '')
                if 'Benchmark' in name:
                    return node
    return None

def find_baseline_method(cls):
    for node in cls.body:
        if not isinstance(node, ast.FunctionDef):
            continue
        for dec in node.decorator_list:
            if not isinstance(dec, ast.Call):
                continue
            fname = dec.func.id if isinstance(dec.func, ast.Name) else getattr(dec.func, 'attr', '')
            if fname != 'register_benchmark':
                continue
            for kw in dec.keywords:
                if kw.arg == 'baseline' and isinstance(kw.value, ast.Constant) and kw.value.value is True:
                    return node
    return None

def has_self_ref(node):
    return any(
        (isinstance(n, ast.Name) and n.id == 'self')
        or (isinstance(n, ast.Attribute) and isinstance(n.value, ast.Name) and n.value.id == 'self')
        for n in ast.walk(node)
    )

def strip_annotations(node):
    for child in ast.walk(node):
        if isinstance(child, ast.arg):
            child.annotation = None
            child.type_comment = None
        elif isinstance(child, ast.FunctionDef):
            child.returns = None
            child.type_comment = None
    return node

def uses_varargs(method):
    return method.args.vararg is not None

def baseline_to_standalone(method):
    if uses_varargs(method):
        return None, '*args baseline — needs manual extraction'

    non_self_args = [strip_annotations(a) for a in method.args.args if a.arg != 'self']
    non_self = ast.arguments(
        posonlyargs=[], args=non_self_args,
        vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]
    )
    body = list(method.body)
    if body and isinstance(body[0], ast.Expr) and isinstance(body[0].value, ast.Constant):
        body = body[1:]  # strip docstring

    # Pattern 1: def _inner(): ...; return _inner
    inner = next((n for n in body if isinstance(n, ast.FunctionDef)), None)
    if inner:
        if any(has_self_ref(n) for n in body if n is not inner):
            return None, 'outer body references self'
        if has_self_ref(inner):
            return None, 'inner function references self'
        new_body = inner.body
    # Pattern 2: return lambda: expr
    elif len(body) == 1 and isinstance(body[0], ast.Return) and isinstance(body[0].value, ast.Lambda):
        lam = body[0].value
        if has_self_ref(lam):
            return None, 'lambda references self'
        new_body = [ast.Return(value=lam.body, lineno=1, col_offset=0)]
    else:
        if any(has_self_ref(n) for n in body):
            return None, 'body references self'
        new_body = body

    new_body = [strip_annotations(n) for n in new_body]
    func = ast.FunctionDef(
        name='op', args=non_self,
        body=new_body or [ast.Pass()],
        decorator_list=[], returns=None,
        lineno=1, col_offset=0
    )
    ast.fix_missing_locations(func)
    code = 'import torch\nimport torch.nn.functional as F\n\n' + ast.unparse(strip_annotations(func))
    return code, None

def op_arg_count_from_code(code):
    tree = ast.parse(code)
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == 'op':
            return len(node.args.posonlyargs) + len(node.args.args)
    return None

def stable_example_id(origin, code, shapes, dtypes):
    payload = json.dumps({
        'origin': origin,
        'pytorch_code': code,
        'input_shapes': shapes,
        'input_dtypes': dtypes,
    }, sort_keys=True)
    return str(uuid.uuid5(uuid.NAMESPACE_URL, 'tried-corpus:' + payload))

def make_record(origin, category, code, shapes, dtypes):
    return {
        'example_id': stable_example_id(origin, code, shapes, dtypes),
        'split': 'train',
        'origin': origin,
        'op_category': category,
        'difficulty': None,
        'pytorch_code': code,
        'input_shapes': shapes,
        'input_dtypes': dtypes,
        'rng_seed': SEED,
    }


## 4. Automatic TritonBench Extraction

This pass extracts simple baseline methods from TritonBench `BenchmarkOperator` wrappers. Anything stateful, vararg-shaped, or dependent on `self` is routed to curation or skipped rather than silently producing broken code.


In [ ]:
# Run extraction
train_records = []
skipped_records = []
manual_review = []
curated_records = []

for op_dir in sorted(OPERATORS_DIR.iterdir()):
    if not op_dir.is_dir():
        continue
    op_name = op_dir.name

    if op_name in SKIP_OPS:
        continue
    if op_name in UNSUPPORTED_TRITONBENCH_OPS:
        skipped_records.append({
            'origin': f'tritonbench/{op_name}',
            'reason': UNSUPPORTED_TRITONBENCH_OPS[op_name],
        })
        continue
    if op_name in CURATED_TRITONBENCH_OPS:
        curated_records.append({'origin': f'tritonbench/{op_name}', 'reason': 'provided by MANUAL_TRITONBENCH'})
        continue

    op_file = op_dir / 'operator.py'
    if not op_file.exists():
        skipped_records.append({'origin': f'tritonbench/{op_name}', 'reason': 'no operator.py'})
        continue

    try:
        tree = ast.parse(op_file.read_text())
    except SyntaxError as e:
        skipped_records.append({'origin': f'tritonbench/{op_name}', 'reason': f'SyntaxError: {e}'})
        continue

    cls = find_benchmark_class(tree)
    if cls is None:
        skipped_records.append({'origin': f'tritonbench/{op_name}', 'reason': 'no BenchmarkOperator subclass'})
        continue

    baseline = find_baseline_method(cls)
    if baseline is None:
        skipped_records.append({'origin': f'tritonbench/{op_name}', 'reason': 'no baseline method'})
        continue

    code, err = baseline_to_standalone(baseline)
    if err:
        manual_review.append({'origin': f'tritonbench/{op_name}', 'reason': err})
        continue

    category = CATEGORY_MAP.get(op_name, 'other')
    shapes, dtypes = OPERATOR_SHAPES.get(op_name, ([[1024, 1024]], ['float32']))
    arg_count = op_arg_count_from_code(code)
    if arg_count != len(shapes):
        manual_review.append({
            'origin': f'tritonbench/{op_name}',
            'reason': f'op expects {arg_count} args but shape table provides {len(shapes)}',
        })
        continue

    train_records.append(make_record(f'tritonbench/{op_name}', category, code, shapes, dtypes))
    print(f'  ✓ {op_name:40s} → {category}')

print(f'\nExtracted:     {len(train_records)}')
print(f'Curated:       {len(curated_records)}')
print(f'Skipped:       {len(skipped_records)}')
print(f'Manual review: {len(manual_review)}')
if manual_review:
    print('\nManual review needed (* means needs hand-written functional version):')
    for r in manual_review:
        print(f'  * {r["origin"]}: {r["reason"]}')


## 5. Curated TritonBench References

Some benchmark wrappers are not directly usable as standalone functions, so this cell provides explicit PyTorch equivalents. Curated rows still pass through the same static and runtime validation as generated rows.


In [ ]:
MANUAL_TRITONBENCH = [
      {
          'origin': 'tritonbench/gemm', 'op_category': 'matmul',
          'input_shapes': [[512, 512], [512, 512], [512]],
          'input_dtypes': ['float16', 'float16', 'float16'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(a, b, bias):\n'
              '    return torch.matmul(a, b) + bias\n'
          ),
      },
      {
          'origin': 'tritonbench/gather_gemv', 'op_category': 'matmul',
          'input_shapes': [[512, 512], [512]],
          'input_dtypes': ['float16', 'float16'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(a, x):\n'
              '    return torch.matmul(a, x)\n'
          ),
      },
      {
          'origin': 'tritonbench/cross_entropy', 'op_category': 'loss',
          'input_shapes': [[1024, 4096], [1024]],
          'input_dtypes': ['float32', 'int64'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(logits, target):\n'
              '    return F.cross_entropy(logits, target % logits.shape[-1])\n'
          ),
      },
      {
          'origin': 'tritonbench/kl_div', 'op_category': 'loss',
          'input_shapes': [[1024, 4096], [1024, 4096]],
          'input_dtypes': ['float32', 'float32'],
          'pytorch_code': (
              'import torch\n'
              'import torch.nn.functional as F\n\n'
              'def op(input, target):\n'
              '    return F.kl_div(F.log_softmax(input, dim=-1), F.softmax(target, dim=-1), reduction="batchmean")\n'
          ),
      },
      {
          'origin': 'tritonbench/jsd', 'op_category': 'loss',
          'input_shapes': [[1024, 4096], [1024, 4096]],
          'input_dtypes': ['float32', 'float32'],
          'pytorch_code': (
              'import torch\n'
              'import torch.nn.functional as F\n\n'
              'def op(p_logits, q_logits):\n'
              '    p = F.softmax(p_logits, dim=-1)\n'
              '    q = F.softmax(q_logits, dim=-1)\n'
              '    m = 0.5 * (p + q)\n'
              '    return 0.5 * (F.kl_div(m.log(), p, reduction="batchmean") + F.kl_div(m.log(), q, reduction="batchmean"))\n'
          ),
      },
      {
          'origin': 'tritonbench/fused_linear_cross_entropy', 'op_category': 'loss',
          'input_shapes': [[1024, 768], [4096, 768], [1024]],
          'input_dtypes': ['float32', 'float32', 'int64'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(x, weight, target):\n'
              '    logits = F.linear(x, weight)\n'
              '    return F.cross_entropy(logits, target % logits.shape[-1])\n'
          ),
      },
      {
          'origin': 'tritonbench/fused_linear_jsd', 'op_category': 'loss',
          'input_shapes': [[1024, 768], [4096, 768], [1024, 4096]],
          'input_dtypes': ['float32', 'float32', 'float32'],
          'pytorch_code': (
              'import torch\n'
              'import torch.nn.functional as F\n\n'
              'def op(x, weight, target_logits):\n'
              '    logits = F.linear(x, weight)\n'
              '    p = F.softmax(logits, dim=-1)\n'
              '    q = F.softmax(target_logits, dim=-1)\n'
              '    m = 0.5 * (p + q)\n'
              '    return 0.5 * (F.kl_div(m.log(), p, reduction="batchmean") + F.kl_div(m.log(), q, reduction="batchmean"))\n'
          ),
      },
      {
          'origin': 'tritonbench/geglu', 'op_category': 'activation',
          'input_shapes': [[32, 128, 4096], [32, 128, 4096]],
          'input_dtypes': ['float16', 'float16'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(x, gate):\n'
              '    return x * F.gelu(gate)\n'
          ),
      },
      {
          'origin': 'tritonbench/swiglu', 'op_category': 'activation',
          'input_shapes': [[32, 128, 4096], [32, 128, 4096]],
          'input_dtypes': ['float16', 'float16'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(x, gate):\n'
              '    return x * F.silu(gate)\n'
          ),
      },
      {
          'origin': 'tritonbench/embedding', 'op_category': 'embedding',
          'input_shapes': [[32, 128], [50257, 768]],
          'input_dtypes': ['int64', 'float16'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(input_ids, weight):\n'
              '    return F.embedding(input_ids, weight)\n'
          ),
      },
      {
          'origin': 'tritonbench/flash_attention', 'op_category': 'fused_attention',
          'input_shapes': [[2, 12, 128, 64], [2, 12, 128, 64], [2, 12, 128, 64]],
          'input_dtypes': ['float16', 'float16', 'float16'],
          'pytorch_code': (
              'import torch, math\n'
              'import torch.nn.functional as F\n\n'
              'def op(q, k, v):\n'
              '    sm_scale = 1.0 / math.sqrt(q.shape[-1])\n'
              '    seq_len = q.shape[2]\n'
              '    M = torch.tril(torch.ones((seq_len, seq_len), device=q.device))\n'
              '    p = torch.matmul(q, k.transpose(2, 3)) * sm_scale\n'
              '    p = p.masked_fill(M == 0, float("-inf"))\n'
              '    p = torch.softmax(p.float(), dim=-1).to(q.dtype)\n'
              '    return torch.matmul(p, v)\n'
          ),
      },
      {
          'origin': 'tritonbench/grouped_gemm', 'op_category': 'matmul',
          'input_shapes': [[4, 128, 512], [4, 512, 256]],
          'input_dtypes': ['float16', 'float16'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(group_A, group_B):\n'
              '    return torch.bmm(group_A, group_B)\n'
          ),
      },
      {
          'origin': 'tritonbench/layer_norm', 'op_category': 'normalization',
          'input_shapes': [[32, 128, 768], [768], [768]],
          'input_dtypes': ['float32', 'float32', 'float32'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(x, weight, bias):\n'
              '    return F.layer_norm(x, [x.shape[-1]], weight, bias)\n'
          ),
      },
      {
          'origin': 'tritonbench/rms_norm', 'op_category': 'normalization',
          'input_shapes': [[32, 128, 4096], [4096]],
          'input_dtypes': ['float32', 'float32'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(x, weight):\n'
              '    variance = x.pow(2).mean(-1, keepdim=True)\n'
              '    return weight * x * torch.rsqrt(variance + 1e-5)\n'
          ),
      },
      {
          'origin': 'tritonbench/inductor_residual_rmsnorm', 'op_category': 'normalization',
          'input_shapes': [[32, 128, 768], [32, 128, 768], [768]],
          'input_dtypes': ['float32', 'float32', 'float32'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(x, residual, weight):\n'
              '    hidden = x + residual\n'
              '    variance = hidden.float().pow(2).mean(-1, keepdim=True)\n'
              '    hidden = hidden * torch.rsqrt(variance + 1e-6)\n'
              '    return weight * hidden\n'
          ),
      },
      {
          'origin': 'tritonbench/rope', 'op_category': 'elementwise_chain',
          'input_shapes': [[2, 32, 128, 64], [2, 32, 128, 64], [128, 64], [128, 64]],
          'input_dtypes': ['float16', 'float16', 'float32', 'float32'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(q, k, cos, sin):\n'
              '    def rotate_half(x):\n'
              '        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]\n'
              '        return torch.cat((-x2, x1), dim=-1)\n'
              '    cos = cos.unsqueeze(0).unsqueeze(0)\n'
              '    sin = sin.unsqueeze(0).unsqueeze(0)\n'
              '    q_out = q * cos + rotate_half(q) * sin\n'
              '    k_out = k * cos + rotate_half(k) * sin\n'
              '    return torch.stack([q_out, k_out])\n'
          ),
      },
      {
          'origin': 'tritonbench/inductor_fused_linear_gelu', 'op_category': 'activation',
          'input_shapes': [[32, 128, 768], [3072, 768], [3072]],
          'input_dtypes': ['float32', 'float32', 'float32'],
          'pytorch_code': (
              'import torch.nn.functional as F\n\n'
              'def op(x, weight, bias):\n'
              '    return F.gelu(F.linear(x, weight, bias))\n'
          ),
      },
      {
          'origin': 'tritonbench/inductor_softmax_fused', 'op_category': 'reduction',
          'input_shapes': [[4096, 1024], [1024]],
          'input_dtypes': ['float16', 'float16'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(x, bias):\n'
              '    return torch.softmax(x + bias, dim=-1)\n'
          ),
      },
      {
          'origin': 'tritonbench/sum', 'op_category': 'reduction',
          'input_shapes': [[4096, 1024]],
          'input_dtypes': ['float32'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(x):\n'
              '    return torch.sum(x, dim=-1)\n'
          ),
      },
      {
          'origin': 'tritonbench/welford', 'op_category': 'reduction',
          'input_shapes': [[4096, 1024]],
          'input_dtypes': ['float32'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(x):\n'
              '    mean = x.mean(dim=-1)\n'
              '    var = x.var(dim=-1, unbiased=False)\n'
              '    return torch.stack([mean, var])\n'
          ),
      },
      {
          'origin': 'tritonbench/vector_exp', 'op_category': 'elementwise_chain',
          'input_shapes': [[1048576]],
          'input_dtypes': ['float32'],
          'pytorch_code': (
              'import torch\n\n'
              'def op(x):\n'
              '    return torch.exp(x)\n'
          ),
      },
  ]

MANUAL_TRITONBENCH = [
    p for p in MANUAL_TRITONBENCH
    if p['origin'].split('/')[-1] not in UNSUPPORTED_TRITONBENCH_OPS
]

manual_origins = {p['origin'] for p in MANUAL_TRITONBENCH}
train_records = [r for r in train_records if r['origin'] not in manual_origins]

for p in MANUAL_TRITONBENCH:
    train_records.append(make_record(
        p['origin'], p['op_category'], p['pytorch_code'],
        p['input_shapes'], p['input_dtypes'],
    ))

print(f'Manual TritonBench records added: {len(MANUAL_TRITONBENCH)}')
print(f'train_records now: {len(train_records)}')
if len(MANUAL_TRITONBENCH) != len(CURATED_TRITONBENCH_OPS):
    missing = sorted(CURATED_TRITONBENCH_OPS - {p['origin'].split('/')[-1] for p in MANUAL_TRITONBENCH})
    extra = sorted({p['origin'].split('/')[-1] for p in MANUAL_TRITONBENCH} - CURATED_TRITONBENCH_OPS)
    print(f'WARNING: curated/manual mismatch. missing={missing}, extra={extra}')


## 6. Curated Training Patterns

These rows deliberately fill categories that model tracing under-represents: reductions, elementwise chains, attention/masking, and embedding/indexing. They use `origin="curated/train/..."` so they cannot be confused with the held-out eval split or `synthetic/fusion` eval rows.


In [ ]:
CURATED_TRAIN_PATTERNS = [
    # Reductions and numerically interesting row-wise summaries.
    {
        'origin': 'curated/train/reduction_logsumexp_fp32', 'op_category': 'reduction',
        'input_shapes': [[2048, 1024]], 'input_dtypes': ['float32'],
        'pytorch_code': 'import torch\n\ndef op(x):\n    return torch.logsumexp(x, dim=-1)\n',
    },
    {
        'origin': 'curated/train/reduction_mean_center_fp32', 'op_category': 'reduction',
        'input_shapes': [[1024, 768]], 'input_dtypes': ['float32'],
        'pytorch_code': 'import torch\n\ndef op(x):\n    mean = x.mean(dim=-1, keepdim=True)\n    return x - mean\n',
    },
    {
        'origin': 'curated/train/reduction_variance_fp32', 'op_category': 'reduction',
        'input_shapes': [[1024, 768]], 'input_dtypes': ['float32'],
        'pytorch_code': 'import torch\n\ndef op(x):\n    mean = x.mean(dim=-1, keepdim=True)\n    return ((x - mean) * (x - mean)).mean(dim=-1)\n',
    },
    {
        'origin': 'curated/train/reduction_softmax_bias_fp16', 'op_category': 'reduction',
        'input_shapes': [[2048, 1024], [1024]], 'input_dtypes': ['float16', 'float16'],
        'pytorch_code': 'import torch\n\ndef op(x, bias):\n    return torch.softmax((x + bias).float(), dim=-1).to(x.dtype)\n',
    },
    {
        'origin': 'curated/train/reduction_l2_norm_fp32', 'op_category': 'reduction',
        'input_shapes': [[4096, 256]], 'input_dtypes': ['float32'],
        'pytorch_code': 'import torch\n\ndef op(x):\n    return torch.sqrt((x * x).sum(dim=-1) + 1e-6)\n',
    },
    {
        'origin': 'curated/train/reduction_max_shift_fp32', 'op_category': 'reduction',
        'input_shapes': [[2048, 512]], 'input_dtypes': ['float32'],
        'pytorch_code': 'import torch\n\ndef op(x):\n    values = x.max(dim=-1, keepdim=True).values\n    return x - values\n',
    },

    # Elementwise chains: useful for fusion practice without hidden state.
    {
        'origin': 'curated/train/elementwise_silu_mul_bias_fp32', 'op_category': 'elementwise_chain',
        'input_shapes': [[32, 128, 2048], [2048]], 'input_dtypes': ['float32', 'float32'],
        'pytorch_code': 'import torch\nimport torch.nn.functional as F\n\ndef op(x, bias):\n    y = x + bias\n    return F.silu(y) * y\n',
    },
    {
        'origin': 'curated/train/elementwise_gelu_residual_fp32', 'op_category': 'elementwise_chain',
        'input_shapes': [[32, 128, 768], [32, 128, 768]], 'input_dtypes': ['float32', 'float32'],
        'pytorch_code': 'import torch\nimport torch.nn.functional as F\n\ndef op(x, residual):\n    return F.gelu(x) + residual * 0.5\n',
    },
    {
        'origin': 'curated/train/elementwise_sigmoid_gate_fp16', 'op_category': 'elementwise_chain',
        'input_shapes': [[32, 128, 2048], [32, 128, 2048]], 'input_dtypes': ['float16', 'float16'],
        'pytorch_code': 'import torch\n\ndef op(x, gate):\n    return x * torch.sigmoid(gate)\n',
    },
    {
        'origin': 'curated/train/elementwise_tanh_affine_fp32', 'op_category': 'elementwise_chain',
        'input_shapes': [[4096, 1024], [1024], [1024]], 'input_dtypes': ['float32', 'float32', 'float32'],
        'pytorch_code': 'import torch\n\ndef op(x, scale, bias):\n    return torch.tanh(x * scale + bias)\n',
    },
    {
        'origin': 'curated/train/elementwise_clamp_square_fp32', 'op_category': 'elementwise_chain',
        'input_shapes': [[1048576]], 'input_dtypes': ['float32'],
        'pytorch_code': 'import torch\n\ndef op(x):\n    y = torch.clamp(x, -3.0, 3.0)\n    return y * y + 0.25 * y\n',
    },
    {
        'origin': 'curated/train/elementwise_rope_pair_fp16', 'op_category': 'elementwise_chain',
        'input_shapes': [[2, 32, 128, 64], [128, 64], [128, 64]], 'input_dtypes': ['float16', 'float32', 'float32'],
        'pytorch_code': 'import torch\n\ndef op(x, cos, sin):\n    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]\n    rotated = torch.cat((-x2, x1), dim=-1)\n    return (x.float() * cos + rotated.float() * sin).to(x.dtype)\n',
    },

    # Attention-like patterns with explicit tensor-only masks/biases.
    {
        'origin': 'curated/train/attention_dense_fp16', 'op_category': 'fused_attention',
        'input_shapes': [[2, 8, 128, 64], [2, 8, 128, 64], [2, 8, 128, 64]],
        'input_dtypes': ['float16', 'float16', 'float16'],
        'pytorch_code': 'import torch, math\n\ndef op(q, k, v):\n    scores = torch.matmul(q, k.transpose(-2, -1)).float() / math.sqrt(q.shape[-1])\n    probs = torch.softmax(scores, dim=-1).to(q.dtype)\n    return torch.matmul(probs, v)\n',
    },
    {
        'origin': 'curated/train/attention_additive_mask_fp16', 'op_category': 'fused_attention',
        'input_shapes': [[2, 8, 128, 64], [2, 8, 128, 64], [2, 8, 128, 64], [1, 1, 128, 128]],
        'input_dtypes': ['float16', 'float16', 'float16', 'float32'],
        'pytorch_code': 'import torch, math\n\ndef op(q, k, v, mask):\n    scores = torch.matmul(q, k.transpose(-2, -1)).float() / math.sqrt(q.shape[-1])\n    scores = scores + mask\n    probs = torch.softmax(scores, dim=-1).to(q.dtype)\n    return torch.matmul(probs, v)\n',
    },
    {
        'origin': 'curated/train/attention_qk_softmax_fp32', 'op_category': 'fused_attention',
        'input_shapes': [[4, 16, 128, 64], [4, 16, 128, 64]],
        'input_dtypes': ['float32', 'float32'],
        'pytorch_code': 'import torch, math\n\ndef op(q, k):\n    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.shape[-1])\n    return torch.softmax(scores, dim=-1)\n',
    },
    {
        'origin': 'curated/train/attention_causal_fp16', 'op_category': 'fused_attention',
        'input_shapes': [[2, 8, 128, 64], [2, 8, 128, 64], [2, 8, 128, 64]],
        'input_dtypes': ['float16', 'float16', 'float16'],
        'pytorch_code': 'import torch, math\n\ndef op(q, k, v):\n    seq = q.shape[-2]\n    causal = torch.tril(torch.ones((seq, seq), device=q.device, dtype=torch.bool))\n    scores = torch.matmul(q, k.transpose(-2, -1)).float() / math.sqrt(q.shape[-1])\n    scores = scores.masked_fill(~causal, float("-inf"))\n    probs = torch.softmax(scores, dim=-1).to(q.dtype)\n    return torch.matmul(probs, v)\n',
    },

    # Embedding/indexing/gather-like tensor-only rows.
    {
        'origin': 'curated/train/embedding_mean_fp16', 'op_category': 'embedding',
        'input_shapes': [[32, 64], [4096, 256]], 'input_dtypes': ['int64', 'float16'],
        'pytorch_code': 'import torch\nimport torch.nn.functional as F\n\ndef op(input_ids, weight):\n    return F.embedding(input_ids, weight).mean(dim=1)\n',
    },
    {
        'origin': 'curated/train/embedding_sum_fp32', 'op_category': 'embedding',
        'input_shapes': [[64, 32], [8192, 128]], 'input_dtypes': ['int64', 'float32'],
        'pytorch_code': 'import torch\nimport torch.nn.functional as F\n\ndef op(input_ids, weight):\n    return F.embedding(input_ids, weight).sum(dim=1)\n',
    },
    {
        'origin': 'curated/train/index_select_rows_fp32', 'op_category': 'embedding',
        'input_shapes': [[4096, 256], [512]], 'input_dtypes': ['float32', 'int64'],
        'pytorch_code': 'import torch\n\ndef op(x, index):\n    return torch.index_select(x, 0, index % x.shape[0])\n',
    },
    {
        'origin': 'curated/train/gather_columns_fp32', 'op_category': 'embedding',
        'input_shapes': [[1024, 512], [1024, 128]], 'input_dtypes': ['float32', 'int64'],
        'pytorch_code': 'import torch\n\ndef op(x, index):\n    return torch.gather(x, 1, index % x.shape[1])\n',
    },
]

curated_train_records = []
for pattern in CURATED_TRAIN_PATTERNS:
    curated_train_records.append(make_record(
        pattern['origin'],
        pattern['op_category'],
        pattern['pytorch_code'],
        pattern['input_shapes'],
        pattern['input_dtypes'],
    ))

curated_by_category = Counter(record['op_category'] for record in curated_train_records)
print(f'Curated training patterns added as candidates: {len(curated_train_records)}')
for category, count in sorted(curated_by_category.items()):
    print(f'  {category:20s} {count}')
print('These are candidates; quota/fingerprint selection decides what reaches validation.')


## 7. Model Subgraph Tracing

This stage traces small, representative windows from transformer and vision models. The goal is pattern diversity, not whole-model reproduction. Failures are reported and do not block the TritonBench corpus.


In [ ]:
from collections import Counter
from torch import fx
from torch.fx.passes.shape_prop import ShapeProp
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    ViTModel,
)
import timm

# First pass: build the extraction scaffold and keep the model list small.
# TinyLlama is the recommended open LLaMA-family model for Colab because
# it is public, lightweight, and exercises modern decoder patterns.
MODEL_SPECS = [
    {
        'name': 'bert-base-uncased',
        'origin': 'hf/bert-base-uncased',
        'family': 'bert',
        'kind': 'encoder',
        'loader': lambda: AutoModel.from_pretrained('bert-base-uncased'),
        'sample_inputs': lambda: {
            'input_ids': torch.randint(0, 30522, (2, 128), device='cuda', dtype=torch.int64),
            'attention_mask': torch.ones((2, 128), device='cuda', dtype=torch.int64),
        },
    },
    {
        'name': 'gpt2',
        'origin': 'hf/gpt2',
        'family': 'gpt2',
        'kind': 'decoder',
        'loader': lambda: AutoModelForCausalLM.from_pretrained('gpt2'),
        'sample_inputs': lambda: {
            'input_ids': torch.randint(0, 50257, (2, 128), device='cuda', dtype=torch.int64),
            'attention_mask': torch.ones((2, 128), device='cuda', dtype=torch.int64),
        },
    },
    {
        'name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'origin': 'hf/tinyllama-1.1b',
        'family': 'llama',
        'kind': 'decoder',
        'loader': lambda: AutoModelForCausalLM.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
        'sample_inputs': lambda: {
            'input_ids': torch.randint(0, 32000, (1, 128), device='cuda', dtype=torch.int64),
            'attention_mask': torch.ones((1, 128), device='cuda', dtype=torch.int64),
        },
    },
    {
        'name': 't5-small',
        'origin': 'hf/t5-small',
        'family': 't5',
        'kind': 'seq2seq',
        'loader': lambda: AutoModelForSeq2SeqLM.from_pretrained('t5-small'),
        'sample_inputs': lambda: {
            'input_ids': torch.randint(0, 32128, (2, 64), device='cuda', dtype=torch.int64),
            'decoder_input_ids': torch.randint(0, 32128, (2, 32), device='cuda', dtype=torch.int64),
            'attention_mask': torch.ones((2, 64), device='cuda', dtype=torch.int64),
        },
    },
    {
        'name': 'google/vit-base-patch16-224',
        'origin': 'hf/vit-base-patch16-224',
        'family': 'vit',
        'kind': 'vision',
        'loader': lambda: ViTModel.from_pretrained('google/vit-base-patch16-224'),
        'sample_inputs': lambda: {
            'pixel_values': torch.randn((2, 3, 224, 224), device='cuda', dtype=torch.float32),
        },
    },
    {
        'name': 'resnet50.a1_in1k',
        'origin': 'timm/resnet50.a1_in1k',
        'family': 'resnet',
        'kind': 'vision',
        'loader': lambda: timm.create_model('resnet50.a1_in1k', pretrained=True),
        'sample_inputs': lambda: {
            'x': torch.randn((2, 3, 224, 224), device='cuda', dtype=torch.float32),
        },
    },
]

SCHEMA_DTYPES = {
    torch.float16: 'float16',
    torch.float32: 'float32',
    torch.bfloat16: 'bfloat16',
    torch.float64: 'float64',
    torch.int8: 'int8',
    torch.int16: 'int16',
    torch.int32: 'int32',
    torch.int64: 'int64',
    torch.bool: 'bool',
}

COMPUTE_TARGETS = {
    torch.matmul, torch.bmm, torch.mm,
    torch.softmax, torch.sum, torch.mean,
    torch.rsqrt, torch.sigmoid, torch.tanh,
}

CALL_FUNCTION_CATEGORY = {
    torch.matmul: 'matmul',
    torch.bmm: 'matmul',
    torch.mm: 'matmul',
    torch.softmax: 'reduction',
    torch.sum: 'reduction',
    torch.mean: 'reduction',
    torch.rsqrt: 'normalization',
    torch.sigmoid: 'activation',
    torch.tanh: 'activation',
}

CALL_METHOD_CATEGORY = {
    'matmul': 'matmul',
    'bmm': 'matmul',
    'softmax': 'reduction',
    'sum': 'reduction',
    'mean': 'reduction',
    'rsqrt': 'normalization',
    'gelu': 'activation',
    'relu': 'activation',
    'sigmoid': 'activation',
    'silu': 'activation',
    'tanh': 'activation',
}

VIEW_LIKE_METHODS = {
    'view', 'reshape', 'permute', 'transpose', 'flatten', 'contiguous',
    'unsqueeze', 'squeeze', 'expand', 'expand_as', 'split', 'chunk',
}

def schema_dtype(tensor):
    dtype = SCHEMA_DTYPES.get(tensor.dtype)
    if dtype is None:
        raise ValueError(f'Unsupported dtype for schema: {tensor.dtype}')
    return dtype

def move_inputs_to_cpu(inputs):
    return {k: v.detach().to('cpu') if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

def module_floating_dtype(module):
    for param in module.parameters(recurse=True):
        if param.is_floating_point():
            return param.dtype
    return torch.float32

def ordered_trace_args(gm, trace_inputs):
    args = []
    for node in gm.graph.nodes:
        if node.op == 'placeholder':
            args.append(trace_inputs[node.target])
    return args

class BertAttentionWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.query = layer.attention.self.query
        self.key = layer.attention.self.key
        self.value = layer.attention.self.value

    def forward(self, hidden_states):
        q = self.query(hidden_states)
        k = self.key(hidden_states)
        v = self.value(hidden_states)
        return q + k + v

class BertMlpWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.dense_in = layer.intermediate.dense
        self.act = layer.intermediate.intermediate_act_fn
        self.dense_out = layer.output.dense

    def forward(self, hidden_states):
        hidden_states = self.dense_in(hidden_states)
        hidden_states = self.act(hidden_states)
        return self.dense_out(hidden_states)

class BertNormWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.norm = layer.output.LayerNorm

    def forward(self, hidden_states):
        return self.norm(hidden_states)

class GPT2AttentionWrapper(torch.nn.Module):
    def __init__(self, block):
        super().__init__()
        self.ln = block.ln_1
        self.c_attn = block.attn.c_attn

    def forward(self, hidden_states):
        hidden_states = self.ln(hidden_states)
        qkv = self.c_attn(hidden_states)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        return torch.tanh(q) + k + v

class GPT2MlpWrapper(torch.nn.Module):
    def __init__(self, block):
        super().__init__()
        self.ln = block.ln_2
        self.c_fc = block.mlp.c_fc
        self.act = block.mlp.act

    def forward(self, hidden_states):
        hidden_states = self.ln(hidden_states)
        hidden_states = self.c_fc(hidden_states)
        hidden_states = self.act(hidden_states)
        return hidden_states + torch.tanh(hidden_states)

class GPT2ProjWrapper(torch.nn.Module):
    def __init__(self, block):
        super().__init__()
        self.c_attn = block.attn.c_attn

    def forward(self, hidden_states):
        qkv = self.c_attn(hidden_states)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        return q + k + v

class LlamaAttentionWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.input_layernorm = layer.input_layernorm
        self.q_proj = layer.self_attn.q_proj
        self.k_proj = layer.self_attn.k_proj
        self.v_proj = layer.self_attn.v_proj
        self.o_proj = layer.self_attn.o_proj

    def forward(self, hidden_states):
        x = hidden_states.float()
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.input_layernorm.variance_epsilon)
        x = x.to(self.q_proj.weight.dtype) * self.input_layernorm.weight.to(self.q_proj.weight.dtype)
        q = self.q_proj(x)
        return self.o_proj(q)

class LlamaMlpWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.post_attention_layernorm = layer.post_attention_layernorm
        self.gate_proj = layer.mlp.gate_proj
        self.up_proj = layer.mlp.up_proj
        self.down_proj = layer.mlp.down_proj
        self.act = layer.mlp.act_fn

    def forward(self, hidden_states):
        x = hidden_states.float()
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.post_attention_layernorm.variance_epsilon)
        hidden_states = x.to(self.gate_proj.weight.dtype) * self.post_attention_layernorm.weight.to(self.gate_proj.weight.dtype)
        gate = self.act(self.gate_proj(hidden_states))
        up = self.up_proj(hidden_states)
        return self.down_proj(gate * up)

class LlamaNormWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.norm = layer.input_layernorm

    def forward(self, hidden_states):
        x = hidden_states.float()
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.norm.variance_epsilon)
        return x.to(hidden_states.dtype) * self.norm.weight

class T5AttentionWrapper(torch.nn.Module):
    def __init__(self, block):
        super().__init__()
        self.layer_norm = block.layer[0].layer_norm
        self.q = block.layer[0].SelfAttention.q
        self.k = block.layer[0].SelfAttention.k
        self.v = block.layer[0].SelfAttention.v
        self.o = block.layer[0].SelfAttention.o

    def forward(self, hidden_states):
        hidden_states = self.layer_norm(hidden_states)
        q = self.q(hidden_states)
        k = self.k(hidden_states)
        v = self.v(hidden_states)
        return self.o(q + k + v)

class T5MlpWrapper(torch.nn.Module):
    def __init__(self, block):
        super().__init__()
        self.layer_norm = block.layer[-1].layer_norm
        dense = block.layer[-1].DenseReluDense
        self.wi = dense.wi
        self.wo = dense.wo
        self.dropout = dense.dropout
        self.act = torch.nn.functional.relu

    def forward(self, hidden_states):
        hidden_states = self.layer_norm(hidden_states)
        hidden_states = self.wi(hidden_states)
        hidden_states = self.act(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return self.wo(hidden_states)

class ViTAttentionWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.layernorm_before = layer.layernorm_before
        self.query = layer.attention.attention.query
        self.key = layer.attention.attention.key
        self.value = layer.attention.attention.value

    def forward(self, hidden_states):
        hidden_states = self.layernorm_before(hidden_states)
        q = self.query(hidden_states)
        k = self.key(hidden_states)
        v = self.value(hidden_states)
        return q + k + v

class ViTMlpWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.layernorm_after = layer.layernorm_after
        self.dense_in = layer.intermediate.dense
        self.act = torch.nn.functional.gelu
        self.dense_out = layer.output.dense

    def forward(self, hidden_states):
        hidden_states = self.layernorm_after(hidden_states)
        hidden_states = self.dense_in(hidden_states)
        hidden_states = self.act(hidden_states)
        return self.dense_out(hidden_states)

class ViTNormWrapper(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.layernorm_before = layer.layernorm_before

    def forward(self, hidden_states):
        return self.layernorm_before(hidden_states)

def pick_block_indices(n_layers):
    if n_layers <= 1:
        return [0]
    mid = n_layers // 2
    last = n_layers - 1
    return sorted({0, mid, last})

def unwrap_model_for_trace(model, spec):
    family = spec['family']
    if family == 'bert':
        modules = {}
        for idx in pick_block_indices(len(model.encoder.layer)):
            layer = model.encoder.layer[idx]
            modules[f'attention_b{idx}'] = BertAttentionWrapper(layer)
            modules[f'mlp_b{idx}'] = BertMlpWrapper(layer)
            modules[f'norm_b{idx}'] = BertNormWrapper(layer)
        return modules
    if family == 'gpt2':
        blocks = model.transformer.h if hasattr(model, 'transformer') else model.h
        modules = {}
        for idx in pick_block_indices(len(blocks)):
            block = blocks[idx]
            modules[f'attention_b{idx}'] = GPT2AttentionWrapper(block)
            modules[f'mlp_b{idx}'] = GPT2MlpWrapper(block)
            modules[f'proj_b{idx}'] = GPT2ProjWrapper(block)
        return modules
    if family == 'llama':
        base = model.model if hasattr(model, 'model') else model
        modules = {}
        for idx in pick_block_indices(len(base.layers)):
            layer = base.layers[idx]
            modules[f'attention_b{idx}'] = LlamaAttentionWrapper(layer)
            modules[f'mlp_b{idx}'] = LlamaMlpWrapper(layer)
            modules[f'norm_b{idx}'] = LlamaNormWrapper(layer)
        return modules
    if family == 't5':
        base = model.encoder if hasattr(model, 'encoder') else model
        modules = {}
        for idx in pick_block_indices(len(base.block)):
            block = base.block[idx]
            modules[f'attention_b{idx}'] = T5AttentionWrapper(block)
            modules[f'mlp_b{idx}'] = T5MlpWrapper(block)
        return modules
    if family == 'vit':
        base = model.vit if hasattr(model, 'vit') else model
        modules = {}
        for idx in pick_block_indices(len(base.encoder.layer)):
            layer = base.encoder.layer[idx]
            modules[f'attention_b{idx}'] = ViTAttentionWrapper(layer)
            modules[f'mlp_b{idx}'] = ViTMlpWrapper(layer)
            modules[f'norm_b{idx}'] = ViTNormWrapper(layer)
        return modules
    if family == 'resnet':
        modules = {}
        stages = [model.layer1, model.layer2, model.layer3]
        for stage_idx, stage in enumerate(stages, start=1):
            modules[f'block_s{stage_idx}'] = stage[0]
        return modules
    return {'full': model}

def build_trace_inputs(module, spec, raw_inputs, trace_kind):
    family = spec['family']
    trace_kind = trace_kind.split('_b')[0]
    if family == 'bert':
        if trace_kind == 'attention':
            width = module.query.in_features
        elif trace_kind == 'mlp':
            width = module.dense_in.in_features
        else:
            width = module.norm.normalized_shape[0]
        hidden = torch.randn((2, 128, width), device='cpu', dtype=module_floating_dtype(module))
        return {'hidden_states': hidden}
    if family == 'gpt2':
        if trace_kind in {'attention', 'mlp'}:
            width = module.ln.normalized_shape[0]
        else:
            # GPT-2 Conv1D stores weights as [in_features, out_features].
            width = module.c_attn.weight.shape[0]
        hidden = torch.randn((2, 128, width), device='cpu', dtype=module_floating_dtype(module))
        return {'hidden_states': hidden}
    if family == 'llama':
        if trace_kind == 'attention':
            norm = module.input_layernorm
        elif trace_kind == 'mlp':
            norm = module.post_attention_layernorm
        else:
            norm = module.norm
        hidden = torch.randn((1, 128, norm.weight.shape[0]), device='cpu', dtype=module_floating_dtype(module))
        return {'hidden_states': hidden}
    if family == 't5':
        width = module.q.weight.shape[1] if trace_kind == 'attention' else module.wi.weight.shape[1]
        hidden = torch.randn((2, 64, width), device='cpu', dtype=module_floating_dtype(module))
        return {'hidden_states': hidden}
    if family == 'vit':
        if trace_kind == 'attention':
            norm = module.layernorm_before
        elif trace_kind == 'mlp':
            norm = module.layernorm_after
        else:
            norm = module.layernorm_before
        hidden = torch.randn((2, 197, norm.normalized_shape[0]), device='cpu', dtype=module_floating_dtype(module))
        return {'hidden_states': hidden}
    if family == 'resnet':
        x = torch.randn((2, module.conv1.in_channels, 56, 56), device='cpu', dtype=module_floating_dtype(module))
        return {'x': x}
    return move_inputs_to_cpu(raw_inputs)

def trace_module(spec):
    model = spec['loader']().eval().to('cuda')
    try:
        raw_inputs = spec['sample_inputs']()
        modules = unwrap_model_for_trace(model, spec)
        traces = {}
        errors = []
        for trace_kind, module in modules.items():
            module = module.eval().to('cpu')
            trace_inputs = build_trace_inputs(module, spec, raw_inputs, trace_kind)
            try:
                gm = fx.symbolic_trace(module, concrete_args={k: v for k, v in trace_inputs.items() if not isinstance(v, torch.Tensor)})
                try:
                    ShapeProp(gm).propagate(*ordered_trace_args(gm, trace_inputs))
                except Exception as shape_err:
                    errors.append(f'{trace_kind}: shape propagation failed ({shape_err})')
                traces[trace_kind] = {'gm': gm, 'trace_inputs': trace_inputs}
            except Exception as e:
                errors.append(f'{trace_kind}: {e}')
        if traces:
            return traces, None if not errors else '; '.join(errors)
        return None, '; '.join(errors) if errors else 'no traces produced'
    finally:
        del model

def is_compute_node(node):
    if node.op == 'call_function':
        return node.target in COMPUTE_TARGETS
    if node.op == 'call_method':
        return node.target not in VIEW_LIKE_METHODS
    if node.op == 'call_module':
        return True
    return False

def node_category(gm, node):
    if node.op == 'call_function':
        return CALL_FUNCTION_CATEGORY.get(node.target, 'elementwise_chain')
    if node.op == 'call_method':
        return CALL_METHOD_CATEGORY.get(node.target, 'elementwise_chain')
    if node.op == 'call_module':
        submodule = gm.get_submodule(node.target)
        name = submodule.__class__.__name__.lower()
        if 'norm' in name:
            return 'normalization'
        if 'embedding' in name:
            return 'embedding'
        if 'conv' in name or 'linear' in name:
            return 'matmul'
        if 'gelu' in name or 'relu' in name or 'silu' in name:
            return 'activation'
    return 'other'

def extract_candidate_windows(gm, min_nodes=2, max_nodes=4):
    compute_nodes = [node for node in gm.graph.nodes if is_compute_node(node)]
    windows = []
    for start in range(len(compute_nodes)):
        for size in range(min_nodes, max_nodes + 1):
            window = compute_nodes[start:start + size]
            if len(window) != size:
                continue
            if not any(node.op in {'call_function', 'call_module'} for node in window):
                continue
            windows.append(window)
    return windows

def summarize_window(gm, window):
    counts = Counter(node_category(gm, node) for node in window)
    category = counts.most_common(1)[0][0] if counts else 'other'
    return {
        'size': len(window),
        'ops': [f'{node.op}:{node.target}' for node in window],
        'op_category': category,
    }

MODEL_TRACE_SUMMARY = []
MODEL_TRACE_CACHE = {}
MODEL_PATTERNS = []
MODEL_TRACE_ERRORS = []

for spec in MODEL_SPECS:
    traces, err = trace_module(spec)
    if err and not traces:
        MODEL_TRACE_ERRORS.append({'origin': spec['origin'], 'reason': err})
        print(f"  ✗ {spec['origin']}: trace failed ({err})")
        continue

    if err:
        MODEL_TRACE_ERRORS.append({'origin': spec['origin'], 'reason': err})

    for trace_kind, traced in traces.items():
        gm = traced['gm']
        trace_inputs = traced['trace_inputs']
        windows = extract_candidate_windows(gm)
        cache_key = f"{spec['origin']}#{trace_kind}"
        MODEL_TRACE_SUMMARY.append({
            'origin': cache_key,
            'n_nodes': sum(1 for _ in gm.graph.nodes),
            'n_windows': len(windows),
            'preview': [summarize_window(gm, w) for w in windows[:5]],
        })
        MODEL_TRACE_CACHE[cache_key] = {
            'gm': gm,
            'trace_inputs': trace_inputs,
            'spec': spec,
            'trace_kind': trace_kind,
            'windows': windows,
        }
print(f'Model traces prepared: {len(MODEL_TRACE_SUMMARY)}')
print(f'Model trace failures: {len(MODEL_TRACE_ERRORS)}')
for summary in MODEL_TRACE_SUMMARY:
    print(f"  ✓ {summary['origin']}: {summary['n_nodes']} fx nodes, {summary['n_windows']} candidate windows")
print('Tracing scaffold ready. Cached graphs are reused in the next cell.')


## 8. Materialize Model Patterns

FX windows are converted into standalone PyTorch functions using narrow emitters. Rejections are counted by reason so missing emitters can be prioritized later.


In [ ]:
MODULE_EMITTERS = {
    'conv2d': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight, bias):',
            '    return F.conv2d(x, weight, bias, stride={stride}, padding={padding}, dilation={dilation}, groups={groups})',
        ],
        'args': lambda mod: {
            'stride': tuple(mod.stride),
            'padding': tuple(mod.padding),
            'dilation': tuple(mod.dilation),
            'groups': mod.groups,
        },
        'shapes': lambda mod: [list(mod.weight.shape), list(mod.bias.shape) if mod.bias is not None else None],
        'dtypes': lambda mod: [schema_dtype(mod.weight), schema_dtype(mod.bias) if mod.bias is not None else None],
        'category': 'convolution',
    },
    'linear': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight, bias):',
            '    return F.linear(x, weight, bias)',
        ],
        'args': lambda mod: {},
        'shapes': lambda mod: [list(mod.weight.shape), list(mod.bias.shape) if mod.bias is not None else [mod.out_features]],
        'dtypes': lambda mod: [schema_dtype(mod.weight), schema_dtype(mod.bias) if mod.bias is not None else schema_dtype(mod.weight)],
        'category': 'matmul',
    },
    'batchnorm2d': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight, bias, running_mean, running_var):',
            '    return F.batch_norm(x, running_mean, running_var, weight, bias, training=False, momentum={momentum}, eps={eps})',
        ],
        'args': lambda mod: {'momentum': mod.momentum, 'eps': mod.eps},
        'shapes': lambda mod: [list(mod.weight.shape), list(mod.bias.shape), list(mod.running_mean.shape), list(mod.running_var.shape)],
        'dtypes': lambda mod: [schema_dtype(mod.weight), schema_dtype(mod.bias), schema_dtype(mod.running_mean), schema_dtype(mod.running_var)],
        'category': 'normalization',
    },
    'layernorm': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight, bias):',
            '    return F.layer_norm(x, {normalized_shape}, weight, bias, eps={eps})',
        ],
        'args': lambda mod: {'normalized_shape': tuple(mod.normalized_shape), 'eps': mod.eps},
        'shapes': lambda mod: [list(mod.weight.shape), list(mod.bias.shape)],
        'dtypes': lambda mod: [schema_dtype(mod.weight), schema_dtype(mod.bias)],
        'category': 'normalization',
    },
    'embedding': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(input_ids, weight):',
            '    return F.embedding(input_ids, weight)',
        ],
        'args': lambda mod: {},
        'shapes': lambda mod: [list(mod.weight.shape)],
        'dtypes': lambda mod: [schema_dtype(mod.weight)],
        'category': 'embedding',
    },
    'relu': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x):',
            '    return F.relu(x, inplace={inplace})',
        ],
        'args': lambda mod: {'inplace': mod.inplace},
        'shapes': lambda mod: [],
        'dtypes': lambda mod: [],
        'category': 'activation',
    },
    'geluactivation': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x):',
            '    return F.gelu(x)',
        ],
        'args': lambda mod: {},
        'shapes': lambda mod: [],
        'dtypes': lambda mod: [],
        'category': 'activation',
    },
    'newgeluactivation': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x):',
            '    return F.gelu(x)',
        ],
        'args': lambda mod: {},
        'shapes': lambda mod: [],
        'dtypes': lambda mod: [],
        'category': 'activation',
    },
    'siluactivation': {
        'template': [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x):',
            '    return F.silu(x)',
        ],
        'args': lambda mod: {},
        'shapes': lambda mod: [],
        'dtypes': lambda mod: [],
        'category': 'activation',
    },
    'llamarmsnorm': {
        'template': [
            'import torch',
            '',
            'def op(x, weight):',
            '    variance = x.pow(2).mean(-1, keepdim=True)',
            '    return weight * x * torch.rsqrt(variance + {eps})',
        ],
        'args': lambda mod: {'eps': mod.variance_epsilon},
        'shapes': lambda mod: [list(mod.weight.shape)],
        'dtypes': lambda mod: [schema_dtype(mod.weight)],
        'category': 'normalization',
    },
}

def placeholder_shapes_and_dtypes(trace_inputs):
    shapes, dtypes = [], []
    for value in trace_inputs.values():
        if isinstance(value, torch.Tensor):
            shapes.append(list(value.shape))
            dtypes.append(schema_dtype(value))
    return shapes, dtypes

def first_tensor_input(trace_inputs):
    return next((v for v in trace_inputs.values() if isinstance(v, torch.Tensor)), None)

def iter_fx_nodes(value):
    if isinstance(value, fx.Node):
        yield value
    elif isinstance(value, (list, tuple)):
        for item in value:
            yield from iter_fx_nodes(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from iter_fx_nodes(item)

def tensor_meta_to_spec(meta):
    if meta is None:
        return None
    if isinstance(meta, (list, tuple)):
        meta = next((m for m in meta if hasattr(m, 'shape') and hasattr(m, 'dtype')), None)
    if meta is None or not hasattr(meta, 'shape') or not hasattr(meta, 'dtype'):
        return None
    dtype = SCHEMA_DTYPES.get(meta.dtype)
    if dtype is None:
        return None
    return list(meta.shape), dtype

def node_input_spec(node, trace_inputs):
    for arg_node in iter_fx_nodes((node.args, node.kwargs)):
        spec = tensor_meta_to_spec(arg_node.meta.get('tensor_meta'))
        if spec is not None:
            return spec
    tensor = first_tensor_input(trace_inputs)
    if tensor is None:
        return None
    return list(tensor.shape), schema_dtype(tensor)

def first_window_input_spec(window, trace_inputs):
    return node_input_spec(window[0], trace_inputs) if window else None

def normalize_module_input_spec(submodule, key, input_spec):
    shape, dtype = list(input_spec[0]), input_spec[1]
    if key == 'linear' and len(shape) >= 1:
        shape[-1] = submodule.weight.shape[1]
        if submodule.weight.is_floating_point():
            dtype = schema_dtype(submodule.weight)
    elif key == 'conv2d' and len(shape) >= 2:
        shape[1] = submodule.weight.shape[1] * submodule.groups
        if submodule.weight.is_floating_point():
            dtype = schema_dtype(submodule.weight)
    elif key == 'batchnorm2d' and len(shape) >= 2:
        shape[1] = submodule.running_mean.shape[0]
        if submodule.running_mean.is_floating_point():
            dtype = schema_dtype(submodule.running_mean)
    elif key == 'layernorm' and len(shape) >= 1:
        normalized_shape = list(submodule.normalized_shape)
        if len(shape) >= len(normalized_shape):
            shape[-len(normalized_shape):] = normalized_shape
        if submodule.weight is not None and submodule.weight.is_floating_point():
            dtype = schema_dtype(submodule.weight)
    return shape, dtype

def pattern_static_error(pattern):
    shapes = pattern['input_shapes']
    code = pattern['pytorch_code']
    if len(shapes) != len(pattern['input_dtypes']):
        return 'shape/dtype length mismatch'
    if 'F.linear(x, weight' in code and len(shapes) >= 2 and shapes[0][-1] != shapes[1][1]:
        return f'linear x last dim {shapes[0][-1]} != weight in_features {shapes[1][1]}'
    if 'F.linear(x, w1' in code and len(shapes) >= 2 and shapes[0][-1] != shapes[1][1]:
        return f'first linear x last dim {shapes[0][-1]} != weight in_features {shapes[1][1]}'
    if 'def op(x, w1, b1, w2, b2):' in code and len(shapes) >= 4:
        if shapes[1][0] != shapes[3][1]:
            return f'second linear in_features {shapes[3][1]} != first linear out_features {shapes[1][0]}'
    if 'F.linear(x, linear_weight' in code:
        weight_idx = 2 if 'def op(x, norm_weight, linear_weight' in code else 3
        if len(shapes) > weight_idx and shapes[0][-1] != shapes[weight_idx][1]:
            return f'norm+linear x last dim {shapes[0][-1]} != weight in_features {shapes[weight_idx][1]}'
    if 'F.conv2d' in code and len(shapes) >= 2:
        expected_c = shapes[1][1]
        if len(shapes[0]) >= 2 and shapes[0][1] != expected_c:
            return f'conv input channels {shapes[0][1]} != weight channels {expected_c}'
    if 'F.batch_norm' in code:
        # For standalone BatchNorm rows, the first corpus input is the tensor
        # being normalized. For fused Conv2d->BatchNorm rows, the first corpus
        # input is the pre-conv tensor, so BN channels must be checked against
        # conv output channels instead. Runtime validation remains the final arbiter.
        if 'F.conv2d' in code and len(shapes) >= 5:
            conv_out_channels = shapes[1][0]
            # conv bias is optional; BN parameter block starts after conv weight
            # and optional conv bias. Detect by comparing available 1D tensors.
            bn_start = 3 if len(shapes) >= 6 and len(shapes[2]) == 1 else 2
            if len(shapes) > bn_start + 2 and conv_out_channels != shapes[bn_start + 2][0]:
                return f'fused conv output channels {conv_out_channels} != running_mean {shapes[bn_start + 2][0]}'
        elif len(shapes) >= 4:
            if len(shapes[0]) >= 2 and shapes[0][1] != shapes[3][0]:
                return f'batchnorm input channels {shapes[0][1]} != running_mean {shapes[3][0]}'
    return None

def emit_call_module_pattern(gm, node, trace_inputs, origin):
    submodule = gm.get_submodule(node.target)
    key = submodule.__class__.__name__.lower()
    spec = MODULE_EMITTERS.get(key)
    if spec is None:
        return None

    input_spec = node_input_spec(node, trace_inputs)
    if input_spec is None:
        return None
    input_shape, input_dtype = normalize_module_input_spec(submodule, key, input_spec)
    shapes, dtypes = [input_shape], [input_dtype]

    if key == 'linear' and submodule.bias is None:
        code = '\n'.join([
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight):',
            '    return F.linear(x, weight, None)',
        ])
        return {
            'origin': origin,
            'op_category': spec['category'],
            'input_shapes': shapes + [list(submodule.weight.shape)],
            'input_dtypes': dtypes + [schema_dtype(submodule.weight)],
            'pytorch_code': code,
        }

    if key == 'conv2d' and submodule.bias is None:
        args = spec['args'](submodule)
        code = '\n'.join([
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight):',
            '    return F.conv2d(x, weight, None, stride={stride}, padding={padding}, dilation={dilation}, groups={groups})',
        ]).format(**args)
        return {
            'origin': origin,
            'op_category': spec['category'],
            'input_shapes': shapes + [list(submodule.weight.shape)],
            'input_dtypes': dtypes + [schema_dtype(submodule.weight)],
            'pytorch_code': code,
        }
    extra_shapes = spec['shapes'](submodule)
    extra_dtypes = spec['dtypes'](submodule)
    for shape, dtype in zip(extra_shapes, extra_dtypes):
        if shape is None or dtype is None:
            continue
        shapes.append(shape)
        dtypes.append(dtype)

    code = '\n'.join(spec['template']).format(**spec['args'](submodule))
    return {
        'origin': origin,
        'op_category': spec['category'],
        'input_shapes': shapes,
        'input_dtypes': dtypes,
        'pytorch_code': code,
    }

def emit_linear_activation_pattern(gm, window, trace_inputs, origin):
    submodules = [gm.get_submodule(node.target) for node in window if node.op == 'call_module']
    if len(submodules) < 2:
        return None
    names = [m.__class__.__name__.lower() for m in submodules]
    input_spec = first_window_input_spec(window, trace_inputs)
    if input_spec is None:
        return None
    input_shape, input_dtype = input_spec

    linear_like = {'linear'}
    activation_like = {'geluactivation', 'newgeluactivation', 'siluactivation', 'relu'}
    norm_like = {'layernorm', 'llamarmsnorm'}

    if names[:2] == ['linear', 'linear']:
        first, second = submodules[:2]
        lines = [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, w1, b1, w2, b2):',
            '    x = F.linear(x, w1, b1)',
            '    return F.linear(x, w2, b2)',
        ]
        return {
            'origin': origin,
            'op_category': 'matmul',
            'input_shapes': [input_shape, list(first.weight.shape), list(first.bias.shape) if first.bias is not None else [first.out_features], list(second.weight.shape), list(second.bias.shape) if second.bias is not None else [second.out_features]],
            'input_dtypes': [input_dtype, schema_dtype(first.weight), schema_dtype(first.bias) if first.bias is not None else schema_dtype(first.weight), schema_dtype(second.weight), schema_dtype(second.bias) if second.bias is not None else schema_dtype(second.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    if names[0] in linear_like and names[1] in activation_like:
        linear, activation = submodules[:2]
        act_expr = 'F.relu(x)' if names[1] == 'relu' else ('F.silu(x)' if names[1] == 'siluactivation' else 'F.gelu(x)')
        lines = [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, weight, bias):',
            '    x = F.linear(x, weight, bias)',
            f'    return {act_expr}',
        ]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape, list(linear.weight.shape), list(linear.bias.shape) if linear.bias is not None else [linear.out_features]],
            'input_dtypes': [input_dtype, schema_dtype(linear.weight), schema_dtype(linear.bias) if linear.bias is not None else schema_dtype(linear.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    if len(names) >= 3 and names[0] in linear_like and names[1] in activation_like and names[2] in linear_like:
        first, activation, second = submodules[:3]
        act_expr = 'F.relu(x)' if names[1] == 'relu' else ('F.silu(x)' if names[1] == 'siluactivation' else 'F.gelu(x)')
        lines = [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, w1, b1, w2, b2):',
            '    x = F.linear(x, w1, b1)',
            f'    x = {act_expr}',
            '    return F.linear(x, w2, b2)',
        ]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape, list(first.weight.shape), list(first.bias.shape) if first.bias is not None else [first.out_features], list(second.weight.shape), list(second.bias.shape) if second.bias is not None else [second.out_features]],
            'input_dtypes': [input_dtype, schema_dtype(first.weight), schema_dtype(first.bias) if first.bias is not None else schema_dtype(first.weight), schema_dtype(second.weight), schema_dtype(second.bias) if second.bias is not None else schema_dtype(second.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    if len(names) >= 3 and names[0] in norm_like and names[1] in linear_like and names[2] in activation_like:
        norm, linear, activation = submodules[:3]
        act_expr = 'F.relu(x)' if names[2] == 'relu' else ('F.silu(x)' if names[2] == 'siluactivation' else 'F.gelu(x)')
        if names[0] == 'llamarmsnorm':
            lines = [
                'import torch',
                'import torch.nn.functional as F',
                '',
                'def op(x, norm_weight, linear_weight, linear_bias):',
                '    variance = x.pow(2).mean(-1, keepdim=True)',
                f'    x = norm_weight * x * torch.rsqrt(variance + {norm.variance_epsilon})',
                '    x = F.linear(x, linear_weight, linear_bias)',
                act_expr.replace('x', 'x').join(['    return ', '']),
            ]
            extra_shapes = [list(norm.weight.shape)]
            extra_dtypes = [schema_dtype(norm.weight)]
        else:
            lines = [
                'import torch',
                'import torch.nn.functional as F',
                '',
                'def op(x, norm_weight, norm_bias, linear_weight, linear_bias):',
                "    x = F.layer_norm(x, {0}, norm_weight, norm_bias, eps={1})".format(tuple(norm.normalized_shape), norm.eps),
                '    x = F.linear(x, linear_weight, linear_bias)',
                act_expr.replace('x', 'x').join(['    return ', '']),
            ]
            extra_shapes = [list(norm.weight.shape), list(norm.bias.shape)]
            extra_dtypes = [schema_dtype(norm.weight), schema_dtype(norm.bias)]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape] + extra_shapes + [list(linear.weight.shape), list(linear.bias.shape) if linear.bias is not None else [linear.out_features]],
            'input_dtypes': [input_dtype] + extra_dtypes + [schema_dtype(linear.weight), schema_dtype(linear.bias) if linear.bias is not None else schema_dtype(linear.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    if len(names) >= 4 and names[0] in norm_like and names[1] in linear_like and names[2] in activation_like and names[3] in linear_like:
        norm, first, activation, second = submodules[:4]
        act_expr = 'F.relu(x)' if names[2] == 'relu' else ('F.silu(x)' if names[2] == 'siluactivation' else 'F.gelu(x)')
        if names[0] == 'llamarmsnorm':
            lines = [
                'import torch',
                'import torch.nn.functional as F',
                '',
                'def op(x, norm_weight, w1, b1, w2, b2):',
                '    variance = x.pow(2).mean(-1, keepdim=True)',
                f'    x = norm_weight * x * torch.rsqrt(variance + {norm.variance_epsilon})',
                '    x = F.linear(x, w1, b1)',
                act_expr.join(['    x = ', '']),
                '    return F.linear(x, w2, b2)',
            ]
            extra_shapes = [list(norm.weight.shape)]
            extra_dtypes = [schema_dtype(norm.weight)]
        else:
            lines = [
                'import torch',
                'import torch.nn.functional as F',
                '',
                'def op(x, norm_weight, norm_bias, w1, b1, w2, b2):',
                "    x = F.layer_norm(x, {0}, norm_weight, norm_bias, eps={1})".format(tuple(norm.normalized_shape), norm.eps),
                '    x = F.linear(x, w1, b1)',
                act_expr.join(['    x = ', '']),
                '    return F.linear(x, w2, b2)',
            ]
            extra_shapes = [list(norm.weight.shape), list(norm.bias.shape)]
            extra_dtypes = [schema_dtype(norm.weight), schema_dtype(norm.bias)]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape] + extra_shapes + [list(first.weight.shape), list(first.bias.shape) if first.bias is not None else [first.out_features], list(second.weight.shape), list(second.bias.shape) if second.bias is not None else [second.out_features]],
            'input_dtypes': [input_dtype] + extra_dtypes + [schema_dtype(first.weight), schema_dtype(first.bias) if first.bias is not None else schema_dtype(first.weight), schema_dtype(second.weight), schema_dtype(second.bias) if second.bias is not None else schema_dtype(second.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    if names[0] in norm_like and len(names) >= 2 and names[1] in linear_like:
        norm, linear = submodules[:2]
        if names[0] == 'llamarmsnorm':
            norm_lines = [
                '    variance = x.pow(2).mean(-1, keepdim=True)',
                f'    x = norm_weight * x * torch.rsqrt(variance + {norm.variance_epsilon})',
            ]
            extra_shapes = [list(norm.weight.shape)]
            extra_dtypes = [schema_dtype(norm.weight)]
            args = 'x, norm_weight, linear_weight, linear_bias'
        else:
            norm_lines = [f'    x = F.layer_norm(x, {tuple(norm.normalized_shape)}, norm_weight, norm_bias, eps={norm.eps})']
            extra_shapes = [list(norm.weight.shape), list(norm.bias.shape)]
            extra_dtypes = [schema_dtype(norm.weight), schema_dtype(norm.bias)]
            args = 'x, norm_weight, norm_bias, linear_weight, linear_bias'
        lines = ['import torch', 'import torch.nn.functional as F', '', f'def op({args}):'] + norm_lines + ['    return F.linear(x, linear_weight, linear_bias)']
        return {
            'origin': origin,
            'op_category': 'normalization',
            'input_shapes': [input_shape] + extra_shapes + [list(linear.weight.shape), list(linear.bias.shape) if linear.bias is not None else [linear.out_features]],
            'input_dtypes': [input_dtype] + extra_dtypes + [schema_dtype(linear.weight), schema_dtype(linear.bias) if linear.bias is not None else schema_dtype(linear.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    return None

def emit_transformer_projection_pattern(gm, window, trace_inputs, origin):
    submodules = [gm.get_submodule(node.target) for node in window if node.op == 'call_module']
    if not submodules:
        return None
    names = [m.__class__.__name__.lower() for m in submodules]
    input_spec = first_window_input_spec(window, trace_inputs)
    if input_spec is None:
        return None
    input_shape, input_dtype = input_spec

    if len(names) >= 3 and names[:3] == ['linear', 'linear', 'linear']:
        first, second, third = submodules[:3]
        if len({first.out_features, second.out_features, third.out_features}) != 1:
            return None
        lines = [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, w1, b1, w2, b2, w3, b3):',
            '    y1 = F.linear(x, w1, b1)',
            '    y2 = F.linear(x, w2, b2)',
            '    y3 = F.linear(x, w3, b3)',
            '    return y1 + y2 + y3',
        ]
        return {
            'origin': origin,
            'op_category': 'matmul',
            'input_shapes': [input_shape, list(first.weight.shape), list(first.bias.shape) if first.bias is not None else [first.out_features], list(second.weight.shape), list(second.bias.shape) if second.bias is not None else [second.out_features], list(third.weight.shape), list(third.bias.shape) if third.bias is not None else [third.out_features]],
            'input_dtypes': [input_dtype, schema_dtype(first.weight), schema_dtype(first.bias) if first.bias is not None else schema_dtype(first.weight), schema_dtype(second.weight), schema_dtype(second.bias) if second.bias is not None else schema_dtype(second.weight), schema_dtype(third.weight), schema_dtype(third.bias) if third.bias is not None else schema_dtype(third.weight)],
            'pytorch_code': '\n'.join(lines),
        }

    return None

def emit_functional_signature_pattern(gm, window, trace_inputs, origin):
    signature = window_signature(gm, window)
    input_spec = first_window_input_spec(window, trace_inputs)
    if input_spec is None:
        return None
    input_shape, input_dtype = input_spec

    if signature[:2] == ['layernorm', 'size']:
        norm_node = next((node for node in window if node.op == 'call_module'), None)
        if norm_node is None:
            return None
        norm = gm.get_submodule(norm_node.target)
        lines = [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, norm_weight, norm_bias):',
            "    x = F.layer_norm(x, {0}, norm_weight, norm_bias, eps={1})".format(tuple(norm.normalized_shape), norm.eps),
            '    return x',
        ]
        return {
            'origin': origin,
            'op_category': 'normalization',
            'input_shapes': [input_shape, list(norm.weight.shape), list(norm.bias.shape)],
            'input_dtypes': [input_dtype, schema_dtype(norm.weight), schema_dtype(norm.bias)],
            'pytorch_code': '\n'.join(lines),
        }

    if signature[-3:] == ['size', 'tanh', 'size'] or signature[-2:] == ['size', 'tanh']:
        lines = [
            'import torch',
            '',
            'def op(x):',
            '    return torch.tanh(x)',
        ]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape],
            'input_dtypes': [input_dtype],
            'pytorch_code': '\n'.join(lines),
        }

    if signature[:2] == ['tanh', 'size'] or signature[:3] == ['tanh', 'size', 'size'] or signature[:3] == ['size', 'tanh', 'tanh'] or signature[:2] == ['tanh', 'tanh']:
        lines = [
            'import torch',
            '',
            'def op(x):',
            '    y = torch.tanh(x)',
            '    return torch.tanh(y)',
        ]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape],
            'input_dtypes': [input_dtype],
            'pytorch_code': '\n'.join(lines),
        }

    if signature[:3] == ['pow', 'mean', 'rsqrt'] or signature[:2] == ['mean', 'rsqrt']:
        lines = [
            'import torch',
            '',
            'def op(x):',
            '    variance = x.pow(2).mean(-1, keepdim=True)',
            '    return torch.rsqrt(variance + 1e-6)',
        ]
        return {
            'origin': origin,
            'op_category': 'normalization',
            'input_shapes': [input_shape],
            'input_dtypes': [input_dtype],
            'pytorch_code': '\n'.join(lines),
        }

    if signature[:2] == ['rsqrt', 'to'] or signature[:3] == ['rsqrt', 'to', 'linear']:
        lines = [
            'import torch',
            '',
            'def op(x):',
            '    variance = x.float().pow(2).mean(-1, keepdim=True)',
            '    out = torch.rsqrt(variance + 1e-6)',
            '    return out.to(x.dtype)',
        ]
        return {
            'origin': origin,
            'op_category': 'normalization',
            'input_shapes': [input_shape],
            'input_dtypes': [input_dtype],
            'pytorch_code': '\n'.join(lines),
        }

    if signature[:2] == ['to', 'linear']:
        # Pure dtype casts are too low-signal for training and are often
        # artifacts around module boundaries, not standalone optimization tasks.
        return None

    return None

def emit_gated_mlp_pattern(gm, window, trace_inputs, origin):
    submodules = [gm.get_submodule(node.target) for node in window if node.op == 'call_module']
    if len(submodules) < 4:
        return None
    names = [m.__class__.__name__.lower() for m in submodules]
    input_spec = first_window_input_spec(window, trace_inputs)
    if input_spec is None:
        return None
    input_shape, input_dtype = input_spec
    if names[0] == 'linear' and names[1] in {'siluactivation', 'geluactivation', 'newgeluactivation'} and names[2] == 'linear' and names[3] == 'linear':
        gate_proj, act_mod, up_proj, down_proj = submodules[:4]
        act_expr = 'F.silu(gate)' if names[1] == 'siluactivation' else 'F.gelu(gate)'
        lines = [
            'import torch',
            'import torch.nn.functional as F',
            '',
            'def op(x, w_gate, b_gate, w_up, b_up, w_down, b_down):',
            '    x = x.to(w_gate.dtype)',
            '    gate = F.linear(x, w_gate, b_gate)',
            f'    gate = {act_expr}',
            '    up = F.linear(x, w_up, b_up)',
            '    fused = gate * up',
            '    return F.linear(fused, w_down, b_down)',
        ]
        return {
            'origin': origin,
            'op_category': 'activation',
            'input_shapes': [input_shape, list(gate_proj.weight.shape), list(gate_proj.bias.shape) if gate_proj.bias is not None else [gate_proj.out_features], list(up_proj.weight.shape), list(up_proj.bias.shape) if up_proj.bias is not None else [up_proj.out_features], list(down_proj.weight.shape), list(down_proj.bias.shape) if down_proj.bias is not None else [down_proj.out_features]],
            'input_dtypes': [input_dtype, schema_dtype(gate_proj.weight), schema_dtype(gate_proj.bias) if gate_proj.bias is not None else schema_dtype(gate_proj.weight), schema_dtype(up_proj.weight), schema_dtype(up_proj.bias) if up_proj.bias is not None else schema_dtype(up_proj.weight), schema_dtype(down_proj.weight), schema_dtype(down_proj.bias) if down_proj.bias is not None else schema_dtype(down_proj.weight)],
            'pytorch_code': '\n'.join(lines),
        }
    return None

def window_signature(gm, window):
    signature = []
    for node in window:
        if node.op == 'call_module':
            signature.append(gm.get_submodule(node.target).__class__.__name__.lower())
        elif node.op == 'call_method':
            signature.append(str(node.target))
        elif node.op == 'call_function':
            signature.append(getattr(node.target, '__name__', str(node.target).split('.')[-1]))
    return signature

def emit_resnet_window_pattern(gm, window, trace_inputs, origin):
    submodules = [gm.get_submodule(node.target) for node in window if node.op == 'call_module']
    if len(submodules) < 2:
        return None
    names = [module.__class__.__name__.lower() for module in submodules]
    if names[:2] != ['conv2d', 'batchnorm2d']:
        return None

    input_spec = first_window_input_spec(window, trace_inputs)
    if input_spec is None:
        return None
    conv, bn = submodules[:2]
    input_shape, input_dtype = normalize_module_input_spec(conv, 'conv2d', input_spec)
    has_relu = len(names) >= 3 and names[2] == 'relu'

    conv_args = {
        'stride': tuple(conv.stride),
        'padding': tuple(conv.padding),
        'dilation': tuple(conv.dilation),
        'groups': conv.groups,
    }
    args = 'x, conv_weight, bn_weight, bn_bias, running_mean, running_var'
    conv_bias_expr = 'None'
    extra_shapes = [list(conv.weight.shape)]
    extra_dtypes = [schema_dtype(conv.weight)]
    if conv.bias is not None:
        args = 'x, conv_weight, conv_bias, bn_weight, bn_bias, running_mean, running_var'
        conv_bias_expr = 'conv_bias'
        extra_shapes.append(list(conv.bias.shape))
        extra_dtypes.append(schema_dtype(conv.bias))

    lines = [
        'import torch',
        'import torch.nn.functional as F',
        '',
        f'def op({args}):',
        '    x = F.conv2d(x, conv_weight, {conv_bias}, stride={stride}, padding={padding}, dilation={dilation}, groups={groups})'.format(conv_bias=conv_bias_expr, **conv_args),
        f'    x = F.batch_norm(x, running_mean, running_var, bn_weight, bn_bias, training=False, momentum={bn.momentum}, eps={bn.eps})',
    ]
    if has_relu:
        lines.append('    return F.relu(x, inplace=False)')
    else:
        lines.append('    return x')

    return {
        'origin': origin,
        'op_category': 'convolution',
        'input_shapes': [input_shape] + extra_shapes + [list(bn.weight.shape), list(bn.bias.shape), list(bn.running_mean.shape), list(bn.running_var.shape)],
        'input_dtypes': [input_dtype] + extra_dtypes + [schema_dtype(bn.weight), schema_dtype(bn.bias), schema_dtype(bn.running_mean), schema_dtype(bn.running_var)],
        'pytorch_code': '\n'.join(lines),
    }

def materialize_patterns_from_graph(gm, trace_inputs, origin):
    patterns = []
    seen = set()
    rejected = Counter()
    for node in gm.graph.nodes:
        if node.op != 'call_module':
            continue
        pattern = emit_call_module_pattern(gm, node, trace_inputs, origin)
        if pattern is None:
            continue
        static_err = pattern_static_error(pattern)
        if static_err:
            rejected['static:' + static_err] += 1
            continue
        fingerprint = (pattern['op_category'], pattern['pytorch_code'], tuple(tuple(s) for s in pattern['input_shapes']))
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        patterns.append(pattern)
    windows = extract_candidate_windows(gm, min_nodes=2, max_nodes=4)
    for window in windows:
        matched = False
        for emitter_name, emitter in [
            ('linear_activation', emit_linear_activation_pattern),
            ('transformer_projection', emit_transformer_projection_pattern),
            ('functional_signature', emit_functional_signature_pattern),
            ('gated_mlp', emit_gated_mlp_pattern),
            ('resnet_window', emit_resnet_window_pattern),
        ]:
            pattern = emitter(gm, window, trace_inputs, origin)
            if pattern is None:
                continue
            matched = True
            static_err = pattern_static_error(pattern)
            if static_err:
                rejected['static:' + static_err] += 1
                continue
            fingerprint = (pattern['op_category'], pattern['pytorch_code'], tuple(tuple(s) for s in pattern['input_shapes']))
            if fingerprint in seen:
                rejected['duplicate_window'] += 1
                continue
            seen.add(fingerprint)
            patterns.append(pattern)
        if not matched:
            rejected['unmatched:' + '->'.join(window_signature(gm, window)[:4])] += 1
            continue
    return patterns, rejected

MODEL_PATTERNS = []
MODEL_PATTERN_REJECTIONS = {}
for origin, cached in MODEL_TRACE_CACHE.items():
    emitted, rejected = materialize_patterns_from_graph(cached['gm'], cached['trace_inputs'], origin)
    MODEL_PATTERNS.extend(emitted)
    MODEL_PATTERN_REJECTIONS[origin] = rejected

model_pattern_summary = Counter(p['origin'].split('#')[0] for p in MODEL_PATTERNS)
for origin, count in sorted(model_pattern_summary.items()):
    print(f'{origin:30s} {count}')

print('Window previews:')
for origin, cached in sorted(MODEL_TRACE_CACHE.items()):
    previews = [window_signature(cached['gm'], window) for window in cached['windows'][:5]]
    print(f'  {origin:30s} {previews}')

print('Top rejection reasons:')
for origin, rejected in sorted(MODEL_PATTERN_REJECTIONS.items()):
    if not rejected:
        continue
    top = ', '.join(f'{k}={v}' for k, v in rejected.most_common(3))
    print(f'  {origin:30s} {top}')

model_records = []
for p in MODEL_PATTERNS:
    model_records.append(make_record(
        origin=p['origin'],
        category=p['op_category'],
        code=p['pytorch_code'],
        shapes=p['input_shapes'],
        dtypes=p['input_dtypes'],
    ))

print(f'Model patterns loaded: {len(model_records)}')


## 9. Parser Opportunity Report

This report turns rejection counters into an emitter backlog. After each run, copy this cell output if we want to decide which parser/emitter to build next.


In [ ]:
parser_rejection_totals = Counter()
unmatched_signature_totals = Counter()
static_rejection_totals = Counter()

for origin, rejected in sorted(MODEL_PATTERN_REJECTIONS.items()):
    for reason, count in rejected.items():
        parser_rejection_totals[reason] += count
        if reason.startswith('unmatched:'):
            unmatched_signature_totals[reason.removeprefix('unmatched:')] += count
        elif reason.startswith('static:'):
            static_rejection_totals[reason.removeprefix('static:')] += count

print('Parser rejection totals:')
if parser_rejection_totals:
    for reason, count in parser_rejection_totals.most_common(20):
        print(f'  {count:4d}  {reason}')
else:
    print('  none')
print()

print('Top unmatched signatures:')
if unmatched_signature_totals:
    for signature, count in unmatched_signature_totals.most_common(20):
        print(f'  {count:4d}  {signature}')
else:
    print('  none')
print()

print('Static rejection totals:')
if static_rejection_totals:
    for reason, count in static_rejection_totals.most_common(20):
        print(f'  {count:4d}  {reason}')
else:
    print('  none')
print()

print('Emitter backlog hints:')
if any(sig.startswith('conv2d->batchnorm2d') for sig in unmatched_signature_totals):
    print('  - Add or debug conv2d+batchnorm(+relu) emitter; ResNet windows are available.')
if any(sig.startswith('batchnorm2d->relu') or sig.startswith('relu->conv2d') or 'identity' in sig for sig in unmatched_signature_totals):
    print('  - ResNet residual/branch windows remain unmatched. Prefer adding a block-level residual emitter only if the tensor-only schema can express the branch inputs cleanly.')
if any('softmax' in sig or 'matmul' in sig for sig in unmatched_signature_totals):
    print('  - Add attention/reduction graph emitters for matmul/softmax windows.')
if any('embedding' in sig or 'gather' in sig or 'index' in sig for sig in unmatched_signature_totals):
    print('  - Add indexing/embedding emitters or choose models with explicit lookup-heavy blocks.')
if any('size' in sig for sig in unmatched_signature_totals):
    print('  - Many size-only windows are harmless graph noise; keep rejecting unless paired with compute ops.')
if not unmatched_signature_totals:
    print('  - No unmatched signatures reported. Next improvements should come from more model families or curated rows.')


## 10. Optional Trace Preview

Use this lightweight preview when adding model families or emitters. It shows the first few candidate signatures per traced graph without changing the corpus.


In [ ]:
for origin, cached in sorted(MODEL_TRACE_CACHE.items()):
    print(f'=== {origin} ===')
    for window in cached['windows'][:8]:
        print(window_signature(cached['gm'], window))
    print()


## 11. Merge, Fingerprint, And Select

Deduplication is intentionally based on normalized code, category, shapes, and dtypes. Rows with the same code body but different shapes are retained, then separately reported as lower-diversity groups.


In [ ]:
def code_hash(code: str) -> str:
    try:
        normalized = ast.unparse(ast.parse(code))
    except SyntaxError:
        normalized = code
    return hashlib.sha256(normalized.encode()).hexdigest()[:16]

def record_signature(record):
    return (
        code_hash(record['pytorch_code']),
        record['op_category'],
        tuple(tuple(shape) for shape in record['input_shapes']),
        tuple(record['input_dtypes']),
    )

def source_prefix(record):
    return record['origin'].split('/')[0]

def dtype_family(dtype):
    if dtype in {'float16', 'bfloat16'}:
        return 'half'
    if dtype in {'float32', 'float64'}:
        return 'float'
    if dtype.startswith('int'):
        return 'int'
    return dtype

def ast_call_fingerprint(code_text):
    try:
        tree = ast.parse(code_text)
    except SyntaxError:
        return ('unparseable',)
    calls = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            func = node.func
            if isinstance(func, ast.Attribute):
                calls.append(func.attr)
            elif isinstance(func, ast.Name):
                calls.append(func.id)
    # Keep order-insensitive call inventory. Exact order is already represented
    # by code_hash; this coarser fingerprint is for behavior saturation.
    return tuple(sorted(Counter(calls).items()))

def behavior_fingerprint(record):
    ranks = tuple(sorted(len(shape) for shape in record['input_shapes']))
    dtype_families = tuple(sorted(dtype_family(dtype) for dtype in record['input_dtypes']))
    return (
        record['op_category'],
        ast_call_fingerprint(record['pytorch_code']),
        ranks,
        dtype_families,
    )

def selection_priority(record):
    prefix = source_prefix(record)
    # Curated rows are designed to fill thin buckets, then TritonBench, then traced model examples.
    return (
        SOURCE_PRIORITY.get(prefix, 9),
        record['op_category'],
        record['origin'],
        code_hash(record['pytorch_code']),
    )

candidate_records = train_records + curated_train_records + model_records
seen_exact = {}
deduped_records = []
dropped_duplicates = []

for record in candidate_records:
    sig = record_signature(record)
    if sig in seen_exact:
        dropped_duplicates.append({
            'origin': record['origin'],
            'duplicate_of': seen_exact[sig]['origin'],
            'reason': 'exact_signature_duplicate',
            'signature': sig,
        })
        continue
    seen_exact[sig] = record
    deduped_records.append(record)

quota_fill = Counter()
fingerprint_fill = Counter()
selected_records = []
selection_rejections = []

for record in sorted(deduped_records, key=selection_priority):
    category = record['op_category']
    quota = CATEGORY_QUOTAS.get(category, TARGET_TRAIN_RECORDS)
    fingerprint = behavior_fingerprint(record)

    if quota_fill[category] >= quota:
        selection_rejections.append({
            'origin': record['origin'],
            'category': category,
            'reason': 'category_quota_full',
            'fingerprint': repr(fingerprint),
        })
        continue
    if fingerprint_fill[fingerprint] >= MAX_ROWS_PER_BEHAVIOR_FINGERPRINT:
        selection_rejections.append({
            'origin': record['origin'],
            'category': category,
            'reason': 'behavior_fingerprint_saturated',
            'fingerprint': repr(fingerprint),
        })
        continue

    selected_records.append(record)
    quota_fill[category] += 1
    fingerprint_fill[fingerprint] += 1

unique_records = selected_records  # Downstream cells validate/export the selected corpus.
selection_rejections_by_reason = Counter(item['reason'] for item in selection_rejections)
selected_by_source = Counter(source_prefix(record) for record in selected_records)
selected_by_category = Counter(record['op_category'] for record in selected_records)

print('Candidate accounting:')
print(f'  TritonBench candidates:      {len(train_records)}')
print(f'  Curated train candidates:    {len(curated_train_records)}')
print(f'  Model-derived candidates:    {len(model_records)}')
print(f'  Total raw candidates:        {len(candidate_records)}')
print(f'  After exact dedup:           {len(deduped_records)}')
print(f'  Exact duplicates removed:    {len(dropped_duplicates)}')
print(f'  Selected for validation:     {len(selected_records)}')
print(f'  Selection rejections:        {len(selection_rejections)}')
print()

print('Selection rejections by reason:')
if selection_rejections_by_reason:
    for reason, count in selection_rejections_by_reason.most_common():
        print(f'  {reason:32s} {count}')
else:
    print('  none')
print()

print('Quota fill:')
for category, quota in CATEGORY_QUOTAS.items():
    selected = selected_by_category.get(category, 0)
    print(f'  {category:20s} {selected:3d} / {quota:<3d} ({selected / quota:5.1%})')
print()

print('Selected rows by source:')
for prefix, count in selected_by_source.most_common():
    print(f'  {prefix:20s} {count}')
print()

print('Top saturated behavior fingerprints:')
for fingerprint, count in fingerprint_fill.most_common(10):
    if count >= MAX_ROWS_PER_BEHAVIOR_FINGERPRINT:
        print(f'  {count}x {fingerprint}')


## 12. Distribution Audit

This audit is a checkpoint before validation. Thin categories, repeated code bodies, or a low total count mean the output should be treated as a seed corpus rather than final training coverage.


In [ ]:
def print_counter(title, counter, total=None):
    print(title)
    if not counter:
        print('  none')
        print()
        return
    for key, count in sorted(counter.items(), key=lambda x: (-x[1], str(x[0]))):
        pct = '' if total is None else f' ({count / total:5.1%})'
        print(f'  {str(key):30s} {count:4d}{pct}')
    print()

def describe_distribution(name, counter, target_min=None):
    total = sum(counter.values())
    if not counter:
        return f'{name}: no records.'
    top_key, top_count = counter.most_common(1)[0]
    parts = [f'{name}: {len(counter)} groups across {total} counted items; largest group is {top_key} with {top_count}.']
    if target_min:
        thin = {key: counter.get(key, 0) for key in target_min if counter.get(key, 0) < target_min[key]}
        if thin:
            parts.append('Thin groups: ' + ', '.join(f'{key}={value} < {target_min[key]}' for key, value in thin.items()) + '.')
    return ' '.join(parts)

by_category = Counter(r['op_category'] for r in unique_records)
by_origin_prefix = Counter(r['origin'].split('/')[0] for r in unique_records)
by_dtype = Counter(dtype for r in unique_records for dtype in r['input_dtypes'])
by_rank = Counter(max(len(shape) for shape in r['input_shapes']) for r in unique_records)
by_origin = Counter(r['origin'] for r in unique_records)
by_code_hash = defaultdict(list)
for record in unique_records:
    by_code_hash[code_hash(record['pytorch_code'])].append(record)

code_body_duplicates = {h: rows for h, rows in by_code_hash.items() if len(rows) > 1}
duplicate_body_rows = sum(len(rows) for rows in code_body_duplicates.values())
behavioral_diversity_estimate = len(unique_records) - sum(len(rows) - 1 for rows in code_body_duplicates.values())
quota_shortfalls = {
    category: CATEGORY_QUOTAS[category] - by_category.get(category, 0)
    for category in CATEGORY_QUOTAS
    if by_category.get(category, 0) < CATEGORY_QUOTAS[category]
}
minimum_shortfalls = {
    category: CATEGORY_MIN_TARGETS[category] - by_category.get(category, 0)
    for category in CATEGORY_MIN_TARGETS
    if by_category.get(category, 0) < CATEGORY_MIN_TARGETS[category]
}

print_counter('By op_category:', by_category, total=len(unique_records))
print_counter('By source prefix:', by_origin_prefix, total=len(unique_records))
print_counter('By dtype occurrence:', by_dtype, total=sum(by_dtype.values()))
print_counter('By max input rank:', by_rank, total=len(unique_records))

print('Corpus size:')
print(f'  Total selected records:          {len(unique_records)}')
print(f'  Raw candidates before selection: {len(candidate_records)}')
print(f'  Target training records:         {TARGET_TRAIN_RECORDS}')
print(f'  Coverage vs target:              {len(unique_records) / TARGET_TRAIN_RECORDS:5.1%}')
print(f'  Code bodies reused:              {len(code_body_duplicates)} groups')
print(f'  Rows in reused code-body groups: {duplicate_body_rows}')
print(f'  Rough behavioral diversity:      {behavioral_diversity_estimate} unique code bodies')
print()

print('Selection pressure:')
print_counter('Selection rejections:', selection_rejections_by_reason, total=max(1, len(selection_rejections)))
print('Minimum target shortfalls:')
if minimum_shortfalls:
    for category, missing in minimum_shortfalls.items():
        print(f'  {category:20s} needs {missing} more to hit minimum {CATEGORY_MIN_TARGETS[category]}')
else:
    print('  none')
print()

print('Interpretation:')
print('  ' + describe_distribution('Categories', by_category, CATEGORY_MIN_TARGETS))
print('  ' + describe_distribution('Sources', by_origin_prefix))
print('  ' + describe_distribution('Dtypes', by_dtype))
if len(unique_records) < TARGET_TRAIN_RECORDS:
    print(f'  This is still below the broad training target by {TARGET_TRAIN_RECORDS - len(unique_records)} rows.')
if selection_rejections_by_reason.get('behavior_fingerprint_saturated'):
    print('  Some rows were rejected because the same behavior fingerprint was already represented enough times. This is intentional diversity control.')
if duplicate_body_rows:
    print('  Reused code bodies remain useful for shape coverage, but behavior-level diversity is the number to watch.')
print()

print('Most repeated origins:')
for origin, count in by_origin.most_common(12):
    if count > 1:
        print(f'  {origin:45s} {count}')
print()

print('Largest reused code-body groups:')
for code_id, rows in sorted(code_body_duplicates.items(), key=lambda item: -len(item[1]))[:8]:
    examples = ', '.join(row['origin'] for row in rows[:4])
    suffix = '' if len(rows) <= 4 else f', ... +{len(rows) - 4} more'
    print(f'  {code_id}: {len(rows)} rows -> {examples}{suffix}')
print()

print('Debug: first 20 selected fingerprints')
for record in unique_records[:20]:
    print(f"  {record['op_category']:18s} {record['origin']:45s} {behavior_fingerprint(record)}")


## 13. Suspicious Pattern Audit

This cell flags code patterns that are legal Python but risky training data: stochastic ops, internal random generation, file/process access, or low-signal casts. Warnings do not automatically delete rows; validation and export policy decide that.


In [ ]:
SUSPICIOUS_PATTERNS = {
    'dropout': ('high', 'stochastic unless represented with an explicit mask or RNG contract'),
    'training=True': ('high', 'training-mode behavior can be stochastic or stateful'),
    'return x.to(weight.dtype)': ('medium', 'pure dtype cast is usually low training signal for normalization'),
    'torch.randn': ('high', 'corpus code should not create random tensors internally'),
    'torch.randint': ('high', 'corpus code should not create random tensors internally'),
    'open(': ('high', 'file I/O is not allowed in corpus functions'),
    'exec(': ('high', 'dynamic execution is not allowed in corpus functions'),
    'eval(': ('high', 'dynamic execution is not allowed in corpus functions'),
    'subprocess': ('high', 'external process calls are not allowed in corpus functions'),
}

suspicious_records = []
for record in unique_records:
    code_text = record['pytorch_code']
    hits = []
    for pattern, (severity, reason) in SUSPICIOUS_PATTERNS.items():
        if pattern in code_text:
            hits.append({'pattern': pattern, 'severity': severity, 'reason': reason})
    if hits:
        suspicious_records.append({'origin': record['origin'], 'category': record['op_category'], 'hits': hits})

suspicious_by_severity = Counter(hit['severity'] for item in suspicious_records for hit in item['hits'])

print(f'Suspicious records: {len(suspicious_records)}')
if suspicious_by_severity:
    print_counter('By severity:', suspicious_by_severity)

if not suspicious_records:
    print('  none')
else:
    for item in suspicious_records[:30]:
        print(f"  {item['origin']} ({item['category']})")
        for hit in item['hits']:
            print(f"    - [{hit['severity']}] {hit['pattern']}: {hit['reason']}")
    if len(suspicious_records) > 30:
        print(f'  ... {len(suspicious_records) - 30} more')

print()
print('Interpretation:')
if not suspicious_records:
    print('  No suspicious source-code patterns were found in the current corpus.')
else:
    print('  Review or remove suspicious rows before using this corpus for training. High severity rows should block a locked export.')


## 14. Static And Runtime Validation

Validation now checks schema shape, imports/calls, deterministic execution, finite tensor outputs, semantic integer ranges, and positive BatchNorm running variance inputs. Set `RUN_INDUCTOR_SANITY=True` before producing a locked corpus.


In [ ]:
# Tolerance policy — mirrors packages/shared/src/shared/verification/tolerance.py
# Values MUST match that module exactly. Update both together; never update one without the other.

TOLERANCE_POLICIES = {
    'default_fp32':   (1e-5, 1e-5),   # general float32 ops
    'default_fp16':   (1e-3, 1e-3),   # fp16 / bfloat16 ops
    'reduction_fp32': (1e-4, 1e-4),   # ops that accumulate across elements, float32
    'reduction_fp16': (5e-3, 5e-3),   # ops that accumulate across elements, fp16/bf16
}

# Categories whose internal accumulation dominates precision loss.
# For these, dtype selects between the two reduction variants.
# Everything else is dtype-driven between default_fp32 and default_fp16.
_REDUCTION_CATS = {'reduction', 'normalization', 'fused_attention', 'loss'}
_LOW_PREC       = {'float16', 'bfloat16'}


def select_policy(record) -> str:
    """Choose the tolerance policy key for a corpus record."""
    cat          = record['op_category']
    has_low_prec = any(dt in _LOW_PREC for dt in record['input_dtypes'])

    if cat in _REDUCTION_CATS:
        return 'reduction_fp16' if has_low_prec else 'reduction_fp32'
    if cat == 'convolution':
        return 'default_fp32'     # fp32-only in current corpus
    return 'default_fp16' if has_low_prec else 'default_fp32'


def _tol_stats(a: torch.Tensor, b: torch.Tensor, atol: float, rtol: float):
    """(ok, max_abs, mean_abs, n_exceed, pct_exceed)"""
    af, bf   = a.float(), b.float()
    diff     = (af - bf).abs()
    max_abs  = diff.max().item()
    mean_abs = diff.mean().item()
    exceed   = diff > (atol + rtol * bf.abs())
    n_exceed = int(exceed.sum().item())
    pct      = 100.0 * n_exceed / a.numel()
    return not exceed.any().item(), max_abs, mean_abs, n_exceed, pct


def tensor_allclose(a: torch.Tensor, b: torch.Tensor, record: dict) -> bool:
    """Silent path — used for the determinism check."""
    if a.shape != b.shape or a.dtype != b.dtype:
        return False
    if not a.is_floating_point():
        return torch.equal(a, b)
    atol, rtol = TOLERANCE_POLICIES[select_policy(record)]
    ok, *_ = _tol_stats(a, b, atol, rtol)
    return ok


def tensor_allclose_verbose(
    a: torch.Tensor, b: torch.Tensor, record: dict, label: str = ''
) -> bool:
    """Verbose path — used for the Inductor sanity check. Prints a full stats line."""
    tag = f'[{label}] ' if label else ''
    if a.shape != b.shape:
        print(f'    {tag}FAIL  shape mismatch  eager={tuple(a.shape)} inductor={tuple(b.shape)}')
        return False
    if a.dtype != b.dtype:
        print(f'    {tag}FAIL  dtype mismatch  eager={a.dtype} inductor={b.dtype}')
        return False
    if not a.is_floating_point():
        ok = torch.equal(a, b)
        print(f'    {tag}{"PASS" if ok else "FAIL"}  int/bool exact match  dtype={a.dtype}')
        return ok
    policy = select_policy(record)
    atol, rtol = TOLERANCE_POLICIES[policy]
    ok, max_abs, mean_abs, n_exceed, pct = _tol_stats(a, b, atol, rtol)
    print(
        f'    {tag}{"PASS" if ok else "FAIL"}'
        f'  policy={policy}  atol={atol:.0e}  rtol={rtol:.0e}'
        f'  max_abs={max_abs:.3e}  mean_abs={mean_abs:.3e}'
        f'  n_exceed={n_exceed} ({pct:.2f}%)'
        f'  shape={tuple(a.shape)}  dtype={a.dtype}'
    )
    return ok


# Print active policy table so the run log shows exactly what thresholds are in effect.
print('=== Tolerance policy ===')
for _pol, (_atol, _rtol) in TOLERANCE_POLICIES.items():
    print(f'  {_pol:17s}  atol={_atol:.0e}  rtol={_rtol:.0e}')
print(f'  reduction cats : {sorted(_REDUCTION_CATS)}')
print(f'  low-prec dtypes: {sorted(_LOW_PREC)}')


In [ ]:
DTYPE_MAP = {
    'float16': torch.float16,
    'float32': torch.float32,
    'bfloat16': torch.bfloat16,
    'float64': torch.float64,
    'int8': torch.int8,
    'int16': torch.int16,
    'int32': torch.int32,
    'int64': torch.int64,
    'bool': torch.bool,
}

FORBIDDEN_IMPORT_ROOTS = {'os', 'sys', 'subprocess', 'socket', 'requests', 'pathlib', 'shutil'}
FORBIDDEN_CALL_NAMES = {'open', 'exec', 'eval', 'compile', '__import__'}

def function_arg_names(code_text):
    tree = ast.parse(code_text)
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == 'op':
            return [arg.arg for arg in node.args.posonlyargs + node.args.args]
    return []

def static_record_errors(record):
    errors = []
    keys = set(record)
    if keys != REQUIRED_RECORD_KEYS:
        errors.append(f'key mismatch: missing={sorted(REQUIRED_RECORD_KEYS - keys)} extra={sorted(keys - REQUIRED_RECORD_KEYS)}')
    if record.get('split') != 'train':
        errors.append(f"split must be train, got {record.get('split')}")
    if record.get('difficulty') is not None:
        errors.append('training difficulty must be null')
    if record.get('op_category') not in ALLOWED_CATEGORIES:
        errors.append(f"unknown op_category: {record.get('op_category')}")
    if len(record.get('input_shapes', [])) != len(record.get('input_dtypes', [])):
        errors.append('input_shapes/input_dtypes length mismatch')
    for dtype in record.get('input_dtypes', []):
        if dtype not in ALLOWED_DTYPES:
            errors.append(f'unknown dtype: {dtype}')
    for shape in record.get('input_shapes', []):
        if not shape or any((not isinstance(dim, int)) or dim <= 0 for dim in shape):
            errors.append(f'invalid shape: {shape}')

    try:
        tree = ast.parse(record['pytorch_code'])
    except SyntaxError as exc:
        return errors + [f'Python parse failed: {exc}']

    arg_count = op_arg_count_from_code(record['pytorch_code'])
    if arg_count != len(record.get('input_shapes', [])):
        errors.append(f'op expects {arg_count} args but record provides {len(record.get("input_shapes", []))}')

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split('.')[0] in FORBIDDEN_IMPORT_ROOTS:
                    errors.append(f'forbidden import: {alias.name}')
        elif isinstance(node, ast.ImportFrom):
            root = (node.module or '').split('.')[0]
            if root in FORBIDDEN_IMPORT_ROOTS:
                errors.append(f'forbidden import: {node.module}')
        elif isinstance(node, ast.Call):
            call_name = None
            if isinstance(node.func, ast.Name):
                call_name = node.func.id
            elif isinstance(node.func, ast.Attribute):
                call_name = node.func.attr
            if call_name in FORBIDDEN_CALL_NAMES:
                errors.append(f'forbidden call: {call_name}')

    if 'dropout' in record['pytorch_code'] or 'training=True' in record['pytorch_code']:
        errors.append('stochastic training-mode op requires explicit RNG/mask metadata')
    if 'return x.to(weight.dtype)' in record['pytorch_code']:
        errors.append('pure dtype-cast row is low-signal and should not enter the training corpus')
    return errors

def integer_high_for_arg(record, arg_name, arg_index):
    origin = record['origin']
    shapes = record['input_shapes']
    if record['op_category'] == 'embedding' and arg_index == 0 and len(shapes) > 1:
        return shapes[1][0]
    if 'target' in arg_name:
        if 'fused_linear' in origin and len(shapes) > 1:
            return shapes[1][0]
        return shapes[0][-1]
    if 'input_id' in arg_name:
        # Embedding rows are handled above when the table shape is present.
        return 32000
    if 'index' in arg_name:
        # Most curated index/gather rows apply modulo inside the op, but keep
        # the synthetic values moderately broad to exercise nontrivial indices.
        return max(128, record['input_shapes'][0][0] if record.get('input_shapes') else 128)
    return 100

def make_inputs(record):
    inputs = []
    arg_names = function_arg_names(record['pytorch_code'])
    for i, (shape, dtype_str) in enumerate(zip(record['input_shapes'], record['input_dtypes'])):
        dtype = DTYPE_MAP[dtype_str]
        arg_name = arg_names[i] if i < len(arg_names) else f'arg{i}'
        if dtype in (torch.int8, torch.int16, torch.int32, torch.int64):
            high = max(1, integer_high_for_arg(record, arg_name, i))
            tensor = torch.randint(0, high, shape, dtype=dtype, device=DEVICE)
        elif dtype == torch.bool:
            tensor = torch.randint(0, 2, shape, dtype=dtype, device=DEVICE)
        elif arg_name == 'running_var':
            tensor = torch.rand(shape, dtype=dtype, device=DEVICE) + 0.5
        else:
            tensor = torch.randn(shape, dtype=dtype, device=DEVICE)
        inputs.append(tensor)
    return inputs

def clone_inputs(inputs):
    return [x.clone() if isinstance(x, torch.Tensor) else x for x in inputs]

def runtime_validate_record(record):
    static_errors = static_record_errors(record)
    if static_errors:
        return False, '; '.join(static_errors)

    namespace = {}
    try:
        exec(record['pytorch_code'], namespace)
    except Exception as exc:
        return False, f'exec failed: {exc}'

    fn = namespace.get('op')
    if fn is None:
        return False, 'no function named op'

    try:
        torch.manual_seed(record['rng_seed'])
        inputs = make_inputs(record)
        with torch.no_grad():
            out1 = fn(*clone_inputs(inputs))
            out2 = fn(*clone_inputs(inputs))
        if not isinstance(out1, torch.Tensor):
            return False, f'output is {type(out1).__name__}, expected Tensor'
        if out1.numel() == 0:
            return False, 'output tensor has zero elements'
        if out1.is_floating_point() and not torch.isfinite(out1).all().item():
            return False, 'output contains NaN or Inf'
        if not tensor_allclose(out1, out2, record):
            return False, 'op is not deterministic for cloned identical inputs'

        if RUN_INDUCTOR_SANITY:
            policy  = select_policy(record)
            atol, rtol = TOLERANCE_POLICIES[policy]
            print(
                f'  sanity  {record["origin"]}'
                f'  |  cat={record["op_category"]}'
                f'  dtypes={record["input_dtypes"]}'
                f'  shapes={record["input_shapes"]}'
                f'  |  policy={policy}  atol={atol:.0e}  rtol={rtol:.0e}'
            )
            compiled = torch.compile(fn, backend='inductor')
            with torch.no_grad():
                eager_out   = fn(*clone_inputs(inputs))
                inductor_out = compiled(*clone_inputs(inputs))
            if not isinstance(inductor_out, torch.Tensor):
                print(f'    FAIL  Inductor returned {type(inductor_out).__name__}, expected Tensor')
                return False, f'Inductor output is {type(inductor_out).__name__}, expected Tensor'
            ok = tensor_allclose_verbose(eager_out, inductor_out, record, label='eager vs inductor')
            if not ok:
                return False, 'eager and Inductor disagree under extraction tolerance'

        return True, None
    except Exception:
        return False, traceback.format_exc().strip().splitlines()[-1]


validated = []
failed    = []

for record in unique_records:
    ok, err = runtime_validate_record(record)
    if ok:
        validated.append(record)
    else:
        failed.append({'origin': record['origin'], 'reason': err})
        print(f'  FAIL  {record["origin"]}  ->  {err}')

print(f'\nValidated: {len(validated)}  /  {len(unique_records)}')
print(f'Failed:    {len(failed)}')

if validated:
    from collections import Counter as _Counter
    print('\nPolicy distribution over validated records:')
    per_policy: dict = {}
    for r in validated:
        p = select_policy(r)
        per_policy.setdefault(p, []).append(r['op_category'])
    for _p, _cats in sorted(per_policy.items()):
        _counts = _Counter(_cats)
        _summary = '  '.join(f'{c}x{n}' for c, n in sorted(_counts.items()))
        print(f'  {_p:15s}: {len(_cats):3d} records  [{_summary}]')

    print('\nop_category x policy breakdown (validated):')
    cat_policy_counts = _Counter((r['op_category'], select_policy(r)) for r in validated)
    for (cat, policy), n in sorted(cat_policy_counts.items(), key=lambda x: (-x[1], x[0])):
        print(f'  {cat:20s}  {policy:17s}: {n:3d}')

if failed:
    print('\nFailed records:')
    for item in failed:
        print(f'  {item["origin"]}  ->  {item["reason"]}')

if RUN_INDUCTOR_SANITY:
    print('\nInductor sanity: ENABLED')
else:
    print('\nInductor sanity: SKIPPED  — set RUN_INDUCTOR_SANITY=True before locking the corpus')


## 15. Quality Gate

The gate summarizes residual risks. During exploration, warnings are allowed. For a locked corpus run, set `STRICT_EXPORT=True` and resolve all warnings first.


In [ ]:
quality_warnings = []
quality_blockers = []

if len(validated) < TARGET_TRAIN_RECORDS:
    quality_warnings.append(f'validated corpus below target: {len(validated)} / {TARGET_TRAIN_RECORDS}')
if suspicious_records:
    high_suspicious = [
        item for item in suspicious_records
        if any(hit['severity'] == 'high' for hit in item['hits'])
    ]
    if high_suspicious:
        quality_blockers.append(f'{len(high_suspicious)} high-severity suspicious records require removal')
    else:
        quality_warnings.append(f'{len(suspicious_records)} suspicious records require review')
if not RUN_INDUCTOR_SANITY:
    quality_warnings.append('Inductor sanity check was not run')
if failed:
    quality_blockers.append(f'{len(failed)} records failed validation')

validated_by_category = Counter(r['op_category'] for r in validated)
for category, minimum in CATEGORY_MIN_TARGETS.items():
    observed = validated_by_category.get(category, 0)
    if observed < minimum:
        quality_warnings.append(f'thin category coverage: {category}={observed} < {minimum}')

if selection_rejections_by_reason.get('behavior_fingerprint_saturated'):
    quality_warnings.append(
        f'{selection_rejections_by_reason["behavior_fingerprint_saturated"]} candidates rejected by behavior-fingerprint saturation'
    )

if quality_blockers:
    corpus_verdict = 'blocked'
elif quality_warnings:
    corpus_verdict = 'usable_seed_only'
else:
    corpus_verdict = 'ready_for_locked_run'

print('Quality gate:')
print(f'  Verdict: {corpus_verdict}')
print(f'  Validated records: {len(validated)}')
print(f'  Failed records:    {len(failed)}')
print(f'  Suspicious rows:   {len(suspicious_records)}')
print(f'  Inductor sanity:   {"enabled" if RUN_INDUCTOR_SANITY else "skipped"}')
print(f'  Selection rejects: {len(selection_rejections)}')
print()

print('Blockers:')
if quality_blockers:
    for blocker in quality_blockers:
        print(f'  - {blocker}')
else:
    print('  none')
print()

print('Warnings:')
if quality_warnings:
    for warning in quality_warnings:
        print(f'  - {warning}')
else:
    print('  none')
print()

print('Interpretation:')
if corpus_verdict == 'blocked':
    print('  Do not export this as training data until blockers are resolved.')
elif corpus_verdict == 'usable_seed_only':
    print('  This corpus is stronger than a raw scrape because it is quota/fingerprint selected, but it is still not a final training corpus.')
    print('  Next improvements should target remaining shortfalls in the quota report and then rerun with RUN_INDUCTOR_SANITY=True.')
else:
    print('  The corpus has no current warnings. For final collection, keep this notebook/config fixed and archive the manifest with the JSONL.')

if STRICT_EXPORT and quality_blockers:
    raise RuntimeError('STRICT_EXPORT=True and quality issues are present')


## 16. Export

The notebook writes the corpus, skipped-row reasons, and a small manifest. The manifest is useful for comparing corpus runs without opening the full JSONL.


In [ ]:
EXPORT_CORPUS = Path('corpus_train.jsonl')
EXPORT_SKIPPED = Path('skipped.jsonl')
EXPORT_MANIFEST = Path('corpus_train_manifest.json')

with EXPORT_CORPUS.open('w') as f:
    for record in validated:
        f.write(json.dumps(record, sort_keys=True) + '\n')

all_skipped = (
    skipped_records
    + [{'origin': r['origin'], 'reason': r['reason']} for r in manual_review]
    + failed
)
with EXPORT_SKIPPED.open('w') as f:
    for record in all_skipped:
        f.write(json.dumps(record, sort_keys=True) + '\n')

manifest = {
    'record_count': len(validated),
    'skipped_count': len(all_skipped),
    'target_train_records': TARGET_TRAIN_RECORDS,
    'category_quotas': CATEGORY_QUOTAS,
    'category_min_targets': CATEGORY_MIN_TARGETS,
    'max_rows_per_behavior_fingerprint': MAX_ROWS_PER_BEHAVIOR_FINGERPRINT,
    'run_inductor_sanity': RUN_INDUCTOR_SANITY,
    'strict_export': STRICT_EXPORT,
    'corpus_verdict': corpus_verdict,
    'quality_blockers': quality_blockers,
    'quality_warnings': quality_warnings,
    'by_category': dict(Counter(r['op_category'] for r in validated)),
    'by_source_prefix': dict(Counter(r['origin'].split('/')[0] for r in validated)),
    'by_dtype_occurrence': dict(Counter(dtype for r in validated for dtype in r['input_dtypes'])),
    'by_max_input_rank': dict(Counter(str(max(len(shape) for shape in r['input_shapes'])) for r in validated)),
    'candidate_count': len(candidate_records),
    'deduped_candidate_count': len(deduped_records),
    'dropped_duplicate_count': len(dropped_duplicates),
    'selection_rejection_count': len(selection_rejections),
    'selection_rejections_by_reason': dict(selection_rejections_by_reason),
    'quota_fill': dict(quota_fill),
    'suspicious_record_count': len(suspicious_records),
    'suspicious_records': suspicious_records,
    'reused_code_body_group_count': len(code_body_duplicates),
    'rows_in_reused_code_body_groups': duplicate_body_rows,
    'behavioral_diversity_estimate': behavioral_diversity_estimate,
}
EXPORT_MANIFEST.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')

print(f'{EXPORT_CORPUS}  -> {len(validated)} records')
print(f'{EXPORT_SKIPPED}       -> {len(all_skipped)} records')
print(f'{EXPORT_MANIFEST} -> audit metadata')
print(f'Verdict: {corpus_verdict}')
print(f'Selected {len(validated)} records from {len(candidate_records)} candidates.')

if IN_COLAB:
    from google.colab import files
    files.download(str(EXPORT_CORPUS))
    files.download(str(EXPORT_SKIPPED))
    files.download(str(EXPORT_MANIFEST))


## Production Migration Backlog

Next steps: move helpers into `packages/shared/src/shared/dataset`, add unit tests for every validator and emitter, make model tracing configurable from a YAML/JSON file, and run the final corpus build with Inductor sanity enabled on the pinned verification machine.

Model/source priorities for the next corpus pass: add lookup-heavy recommender-style modules, segmentation/detection heads for reductions, a lightweight ConvNeXt/MobileNet path for depthwise and pointwise convs, and an export/Dynamo fallback for seq2seq/control-flow models such as T5. Do not add more BERT/GPT layers until the quota report says matmul/normalization coverage is actually thin.
